# Infinite OCR — DOCX comparison

Scanned PDF (any script, mixed Indic + English) → **one continuous, structure-aware DOCX**.

Two methods on the SAME input, same output format, measured end-to-end:

| | Method A | Method B |
|---|---|---|
| OCR | Bodhan primary + Gemini fallback (per page) | Gemini reads page images directly (chunk-wise) |
| Structure | Nemotron VLM orders boxes + dedup; Python builds DOCX | Gemini returns typed blocks + continuation flags directly |
| Cross-page | Python regex (continuation detector) | LLM handles within chunk; Python stitches between chunks |
| Granularity | 1 call per page per stage | 1 call per CHUNK (default 3 pages) |

**Metrics**: tokens, latency, cost per stage, logged automatically for every HTTP call.

Run cells top to bottom, then jump to the Comparison section at the end.


In [64]:
# ─── CELL 1: setup ────────────────────────────────────────────────────────
import base64, io, json, os, re, sys, time, string, datetime
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pymupdf
import requests
from PIL import Image, ImageDraw, ImageFont

from docx import Document
from docx.shared import Pt, Inches
from html import unescape

# ── INPUT PDF ──
PDF_PATH   = Path("pdf/eng_qp_scanned.pdf")   # change this
PAGES      = -1  #[0, 1, 2, 3, 4, 5, 6, 7, 8]          # -1 = all; or a list of 0-indexed pages e.g. [0, 2, 5]
DPI        = 200
CHUNK_SIZE = 3           # Method B: pages per Gemini call

# ── OUTPUT DIRS ──
OUT_DIR      = Path("compare_out")
METHOD_A_DIR = OUT_DIR / "A_bodhan_nemotron"
METHOD_B_DIR = OUT_DIR / "B_gemini_chunks"
CROP_DIR_A   = METHOD_A_DIR / "crops"
PAGE_DIR_A   = METHOD_A_DIR / "pages"
for d in (OUT_DIR, METHOD_A_DIR, METHOD_B_DIR, CROP_DIR_A, PAGE_DIR_A):
    d.mkdir(parents=True, exist_ok=True)

# ── CREDS ──
BODHAN_KEY     = os.environ.get("BODHAN_OCR_API_KEY")
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")

# ── MODELS ──
OCR_FALLBACK_MODEL = "google/gemini-2.5-flash"              # Method A OCR fallback
ORDER_MODELS       = ["qwen/qwen3-vl-30b-a3b-instruct",       # Method A ordering (primary)
                      "google/gemini-2.5-flash"]            # Method A ordering (fallback)
METHOD_B_MODEL     = "google/gemini-2.5-flash"              # Method B chunk-wise

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BODHAN_URL     = "https://api.bodhan.ai/v1/chat/completions"

# ── PRICING (USD per 1M tokens: input, output) ──
PRICES_USD_PER_M = {
    "google/gemini-2.5-flash":       (0.30, 2.50),
    "nvidia/nemotron-3.5-super-vl":  (0.10, 0.40),   # free on OpenRouter at time of writing
}
BODHAN_USD_PER_PAGE = 0.0024  # fill in your Bodhan per-page rate, or leave None (shown as unpriced)

# ── open PDF, resolve page selection ──
doc  = pymupdf.open(PDF_PATH)
stem = PDF_PATH.stem
total = len(doc)
page_indices = list(range(total)) if PAGES == -1 else [p for p in PAGES if 0 <= p < total]
if not page_indices:
    raise ValueError(f"no valid pages in PAGES={PAGES} for a {total}-page PDF")

print(f"PDF:    {PDF_PATH.name} ({total} pages)")
print(f"range:  processing {len(page_indices)} page(s): {page_indices}")
print(f"chunk:  Method B uses {CHUNK_SIZE} page(s) per Gemini call")
print(f"out:    {OUT_DIR}/")


PDF:    eng_qp_scanned.pdf (19 pages)
range:  processing 19 page(s): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
chunk:  Method B uses 3 page(s) per Gemini call
out:    compare_out/


In [65]:
# ─── CELL 2: metrics instrumentation (wraps requests.post) ────────────────
# Every API call to Bodhan / OpenRouter is logged with model, tokens, latency, cost.
# Call set_stage(method, stage) before each pipeline step so calls are attributed.

CALLS = []                       # one dict per HTTP call
_CTX  = {"method": "-", "stage": "-"}
_stage_t0 = {"method": "-", "stage": "-", "t": None}

def set_stage(method, stage):
    """Tag every HTTP call until the next set_stage(...) with (method, stage)."""
    global _stage_t0
    # close previous stage
    if _stage_t0["t"] is not None:
        CALLS.append({"method": _stage_t0["method"], "stage": _stage_t0["stage"],
                      "engine": "_wallclock_", "ok": True,
                      "tok_in": 0, "tok_out": 0, "cost": 0.0,
                      "seconds": time.perf_counter() - _stage_t0["t"]})
    _stage_t0.update(method=method, stage=stage, t=time.perf_counter())
    _CTX.update(method=method, stage=stage)

def reset_metrics():
    CALLS.clear(); _CTX.update(method="-", stage="-")
    _stage_t0.update(method="-", stage="-", t=None)
    print("[metrics] cleared")

def _extract(url, body_json, resp_json, dt, ok):
    model = "?"
    if isinstance(body_json, dict):
        model = body_json.get("model", model)
    if isinstance(resp_json, dict):
        model = resp_json.get("model") or model
    if model == "?" and "bodhan" in url.lower():
        model = "bodhan/indic-ocr"
    in_tok, out_tok, cost = 0, 0, None
    if isinstance(resp_json, dict):
        u = resp_json.get("usage") or {}
        in_tok  = u.get("prompt_tokens",     0) or 0
        out_tok = u.get("completion_tokens", 0) or 0
        if u.get("cost") is not None:
            cost = float(u["cost"])
    if cost is None:
        for key, (pin, pout) in PRICES_USD_PER_M.items():
            if key in model.lower():
                cost = (in_tok * pin + out_tok * pout) / 1_000_000
                break
    if cost is None and "bodhan" in model.lower() and BODHAN_USD_PER_PAGE is not None:
        cost = float(BODHAN_USD_PER_PAGE)
    return {"method": _CTX["method"], "stage": _CTX["stage"], "engine": model,
            "ok": ok, "tok_in": in_tok, "tok_out": out_tok,
            "cost": cost, "seconds": dt}

# install wrapper (idempotent — safe to re-run this cell)
if not getattr(requests.post, "_instrumented", False):
    _orig_post = requests.post

    def _tracked_post(url, *args, **kwargs):
        # ask OpenRouter to include cost in the usage block
        if "openrouter.ai" in url and isinstance(kwargs.get("json"), dict):
            kwargs["json"] = {**kwargs["json"], "usage": {"include": True}}
        t0 = time.perf_counter()
        body = kwargs.get("json")
        try:
            resp = _orig_post(url, *args, **kwargs)
        except Exception as e:
            dt = time.perf_counter() - t0
            CALLS.append(_extract(url, body, None, dt, ok=False))
            raise
        dt = time.perf_counter() - t0
        try:    data = resp.json()
        except Exception: data = None
        CALLS.append(_extract(url, body, data, dt, ok=resp.ok))
        return resp

    _tracked_post._instrumented = True
    requests.post = _tracked_post
    print("[metrics] requests.post wrapped — call set_stage(method, stage) before each step")
else:
    print("[metrics] already wrapped — this cell is idempotent")


[metrics] already wrapped — this cell is idempotent


## Shared helpers

Used by both methods: preprocessing, text sanitisers (HTML + LaTeX stripping), 
cross-page continuation detector, and the DOCX builder.

The DOCX builder takes a `page_dicts` list (one per page, each with ordered 
`regions`), plus optional `cross_joins` for sentences that span page breaks. 
Both methods produce output in this same intermediate format so the builder 
is shared.


In [66]:
# ─── CELL 4: shared helpers ──────────────────────────────────────────────
# Both methods route through these. Keep this cell focused on reusable pieces;
# method-specific logic lives in Methods A and B below.

# ── preprocess: illumination + CLAHE (BGR-native) ─────────────────────────
def _illumination_normalize(bgr, sigma=35):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    bg = cv2.GaussianBlur(gray, (0, 0), sigma)
    norm = cv2.divide(gray, bg, scale=192).clip(0, 255).astype(np.uint8)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = norm
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def _clahe(bgr, clip=3.0):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def render_page(pno):
    pix = doc[pno].get_pixmap(dpi=DPI, alpha=False)
    return cv2.imdecode(np.frombuffer(pix.tobytes("png"), np.uint8), cv2.IMREAD_COLOR)

# def preprocess_page(pno):
#     return _clahe(_illumination_normalize(render_page(pno)))


def preprocess_page(pno, method="original"):
    bgr = render_page(pno)

    if method == "original":
        # Preserve the scanned page's original appearance.
        return bgr

    if method == "gray":
        # Remove color variation without enhancing noise.
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    if method == "clahe_mild":
        # Mild local contrast enhancement, without background division.
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(
            clipLimit=1.5,
            tileGridSize=(8, 8)
        ).apply(l)
        enhanced = cv2.merge((l, a, b))
        return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    raise ValueError(f"Unknown preprocessing method: {method}")


# ── sanitisers: strip HTML tags and LaTeX math from OCR output ────────────
_HTML_TAG_RE = re.compile(r"<[^>]+>")

def _strip_html(text):
    if not text: return ""
    s = text
    s = re.sub(r"<br\s*/?>",   "\n", s, flags=re.I)
    s = re.sub(r"</tr\s*>",    "\n", s, flags=re.I)
    s = re.sub(r"</t[dh]\s*>", "\n", s, flags=re.I)
    s = _HTML_TAG_RE.sub("", s)
    s = unescape(s)
    s = re.sub(r"[ \t]+\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

_SUPER = str.maketrans("0123456789+-=()n", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ")
_SUB   = str.maketrans("0123456789+-=()",  "₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎")
_LATEX_CMDS = {
    r"\alpha":"α", r"\beta":"β", r"\gamma":"γ", r"\delta":"δ", r"\epsilon":"ε",
    r"\theta":"θ", r"\lambda":"λ", r"\mu":"μ", r"\pi":"π", r"\rho":"ρ",
    r"\sigma":"σ", r"\tau":"τ", r"\phi":"φ", r"\omega":"ω",
    r"\Gamma":"Γ", r"\Delta":"Δ", r"\Theta":"Θ", r"\Lambda":"Λ",
    r"\Pi":"Π", r"\Sigma":"Σ", r"\Phi":"Φ", r"\Omega":"Ω",
    r"\infty":"∞", r"\times":"×", r"\cdot":"·", r"\div":"÷", r"\pm":"±",
    r"\leq":"≤", r"\geq":"≥", r"\neq":"≠", r"\approx":"≈",
    r"\rightarrow":"→", r"\partial":"∂", r"\sum":"∑", r"\int":"∫", r"\sqrt":"√",
    r"\degree":"°", r"\circ":"°",
    r"\overline":"", r"\bar":"", r"\hat":"", r"\vec":"",
    r"\text":"", r"\mathrm":"", r"\mathbf":"", r"\mathit":"",
    r"\left":"", r"\right":"",
    r"\,":" ", r"\;":" ", r"\!":"", r"\ ":" ",
}

def _sup(b):
    return b.translate(_SUPER) if b and all(c in "0123456789+-=()n" for c in b) else f"^({b})"
def _sub(b):
    return b.translate(_SUB)   if b and all(c in "0123456789+-=()"  for c in b) else f"_({b})"

def _flatten_latex(s):
    s = re.sub(r"(\d)\\frac", r"\1 \\frac", s)
    for k, v in _LATEX_CMDS.items(): s = s.replace(k, v)
    s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", r"\1/\2", s)
    s = re.sub(r"\^\{([^{}]*)\}",  lambda m: _sup(m.group(1)), s)
    s = re.sub(r"\^([A-Za-z0-9])", lambda m: _sup(m.group(1)), s)
    s = re.sub(r"_\{([^{}]*)\}",   lambda m: _sub(m.group(1)), s)
    s = re.sub(r"_([A-Za-z0-9])",  lambda m: _sub(m.group(1)), s)
    s = re.sub(r"\\[A-Za-z]+", "", s)
    return s.replace("{", "").replace("}", "").strip()

def _strip_latex(text):
    if not text: return ""
    text = re.sub(r"\$\$(.+?)\$\$", lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    text = re.sub(r"\$(.+?)\$",     lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    return text

def sanitize(text):
    return _strip_latex(_strip_html(text or ""))


# ── cross-page continuation detector (shared) ─────────────────────────────
_TERMINAL_PUNCT_RE = re.compile(r"[.!?।॥:;\"')\]\}]\s*$")
_FURNITURE_RE = re.compile(
    r"^\s*("
    r"scanned\s+(by|with)\s+.+"
    r"|page\s+\d+(\s+of\s+\d+)?"
    r"|\[?\s*p\.?\s*t\.?\s*o\.?\s*\]?"
    r"|\d+-\d+-\d+"
    r"|\[\s*\]"
    r")\s*$", re.I)

_BODY_TYPES = {"paragraph", "list_item"}

def _is_real_body(region):
    if region.get("type") not in _BODY_TYPES: return False
    t = (region.get("text") or "").strip()
    if not t: return False
    return not _FURNITURE_RE.match(t)

def cross_page(page_dicts):
    """Return [(last_id, first_id), ...] to merge across page breaks."""
    joins = []
    print(f"\n[CROSS-PAGE] checking {max(0, len(page_dicts)-1)} page-break(s)")
    for i in range(len(page_dicts) - 1):
        pa, pb = page_dicts[i]["page"], page_dicts[i+1]["page"]
        last, first = None, None
        for r in reversed(page_dicts[i]["regions"]):
            if _is_real_body(r): last = r; break
        for r in page_dicts[i+1]["regions"]:
            if _is_real_body(r): first = r; break
        if last is None or first is None:
            print(f"  p{pa}\u2192p{pb}: NO JOIN (last body={last is not None}, first body={first is not None})")
            continue
        last_text  = (last.get("text") or "").strip()
        first_text = (first.get("text") or "").strip()
        if _TERMINAL_PUNCT_RE.search(last_text):
            print(f"  p{pa}\u2192p{pb}: NO JOIN (last ends with punct). "
                  f"last_tail={last_text[-40:]!r}  first_head={first_text[:40]!r}")
            continue
        print(f"  p{pa}\u2192p{pb}: JOIN   last={last['id']}  first={first['id']}")
        print(f"           last_tail  = {last_text[-60:]!r}")
        print(f"           first_head = {first_text[:60]!r}")
        joins.append((last["id"], first["id"]))
    print(f"[CROSS-PAGE] total joins: {len(joins)}\n")
    return joins


# ── DOCX builder (shared; handles both methods' intermediate format) ──────
_FONT_NAME = "Nirmala UI"

def _apply_font(run, name=_FONT_NAME):
    run.font.name = name
    try:
        from docx.oxml.ns import qn
        from docx.oxml import OxmlElement
        rPr = run._element.get_or_add_rPr()
        rFonts = rPr.find(qn("w:rFonts"))
        if rFonts is None:
            rFonts = OxmlElement("w:rFonts"); rPr.insert(0, rFonts)
        for a in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
            rFonts.set(qn(a), name)
    except Exception:
        pass

def _add_table(doc, rows):
    if not rows: return
    ncols = max(len(r) for r in rows)
    table = doc.add_table(rows=len(rows), cols=ncols)
    try: table.style = "Light Grid Accent 1"
    except KeyError: pass
    for ri, row in enumerate(rows):
        for ci, cell in enumerate(row):
            tc = table.rows[ri].cells[ci]
            tc.text = "" if cell is None else str(cell)
            for p in tc.paragraphs:
                for run in p.runs: _apply_font(run)

def build_docx(page_dicts, cross_joins, out_path):
    """Intermediate format per region:
       {id, type, text?, image_path?, rows?, level?}
       type ∈ {title, heading, paragraph, list_item, table, picture,
               picture_description, caption, header, footer}
    """
    docx = Document()
    docx.styles["Normal"].font.name = _FONT_NAME
    docx.styles["Normal"].font.size = Pt(11)

    merge_into = {first_id: last_id for last_id, first_id in cross_joins}
    para_by_id = {}

    for page in page_dicts:
        print(f"\n[BUILD-DOCX] page {page['page']}: {len(page['regions'])} region(s)")
        for region in page["regions"]:
            rid   = region["id"]
            rtype = region.get("type", "paragraph")

            if rtype == "picture" and region.get("image_path"):
                try:
                    docx.add_picture(region["image_path"], width=Inches(5.5))
                    print(f"  {rid:>18}  PICTURE  path={region['image_path']}")
                except Exception as e:
                    docx.add_paragraph(f"[picture failed: {e}]")
                    print(f"  {rid:>18}  PICTURE-FAIL {e}")
                continue

            if rtype == "table" and region.get("rows"):
                _add_table(docx, region["rows"])
                print(f"  {rid:>18}  TABLE  rows={len(region['rows'])}")
                continue

            text = sanitize(region.get("text") or "")
            if not text:
                raw = (region.get("text") or "").strip()
                print(f"  {rid:>18}  DROP-EMPTY  type={rtype:<12} raw={raw[:60]!r}")
                continue

            if rid in merge_into and merge_into[rid] in para_by_id:
                prev = para_by_id[merge_into[rid]]
                print(f"  {rid:>18}  MERGE-INTO {merge_into[rid]}  "
                      f"appended={text[:60]!r}")
                run = prev.add_run(" " + text); _apply_font(run)
                continue
            print(f"  {rid:>18}  APPEND     type={rtype:<12} text={text[:70]!r}")

            if rtype == "title":
                p = docx.add_heading(text, level=0)
            elif rtype == "heading":
                p = docx.add_heading(text, level=min(3, max(1, region.get("level", 1))))
            elif rtype == "list_item":
                p = docx.add_paragraph(text, style="List Bullet")
            elif rtype in ("header", "footer"):
                p = docx.add_paragraph()
                r = p.add_run(text); r.italic = True; r.font.size = Pt(9)
            elif rtype == "caption":
                p = docx.add_paragraph()
                r = p.add_run(text); r.italic = True
            elif rtype == "picture_description":
                p = docx.add_paragraph()
                r = p.add_run(f"[Figure: {text}]"); r.italic = True
            else:
                p = docx.add_paragraph(text)

            for run in p.runs: _apply_font(run)
            para_by_id[rid] = p

    docx.save(str(out_path))
    return str(out_path)

def save_docx_with_fallback(page_dicts, cross_joins, path):
    try:
        return build_docx(page_dicts, cross_joins, path)
    except PermissionError:
        ts  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        alt = path.with_name(f"{path.stem}_{ts}{path.suffix}")
        print(f"  '{path.name}' locked (open in Word?) → saving as '{alt.name}'")
        return build_docx(page_dicts, cross_joins, alt)

print("shared helpers loaded")


shared helpers loaded


## Method A — Bodhan OCR → Nemotron ordering → DOCX

Per-page pipeline, EXACTLY as in your working DOCX notebook:

1. **OCR** — Bodhan primary (specialist Indic engine); Gemini-flash fallback if Bodhan errors
2. **Dedup** — Python-side geometric dedup (IoU + text subsumption) removes Bodhan's coarse+fine box duplicates
3. **Label** — draw B0, B1, ... outside each box on the preprocessed image
4. **Order** — Nemotron 3.5 Super VL (cheap VLM) reads image + JSON, returns `{order, dropped, layout}`; Gemini fallback if it fails
5. **DOCX** — Python stitches regions in the model's order, applies cross-page continuation, writes `.docx`

The PLAN_PROMPT below is your working version — don't change it unless you want to re-tune.


In [67]:
# ─── CELL 6: Method A ─────────────────────────────────────────────────────
reset_metrics()

# ── debug helpers (print only; no logic change) ─────────────────────────
def _preview(text, n=70):
    if not text: return ""
    return text.strip()[:n].replace("\n", " \u23ce ")

def _show_block(prefix, b, label=None):
    lbl = label or b.get("_label") or f"B#{b.get('order','?')}"
    typ = b.get("type", "?")
    bbox = b.get("bbox_xyxy")
    bbox_s = "null" if bbox is None else f"[{bbox[0]:.0f},{bbox[1]:.0f},{bbox[2]:.0f},{bbox[3]:.0f}]"
    print(f"    {prefix} {lbl:>6}  type={typ:<14}  bbox={bbox_s:<25}  text={_preview(b.get('text'))!r}")


# ─────────────────────────────────────────────────────────────────────────
# A.1  OCR  —  Bodhan primary + Gemini fallback (unchanged)
# ─────────────────────────────────────────────────────────────────────────
def _b64_png(bgr):
    return base64.b64encode(cv2.imencode(".png", bgr)[1].tobytes()).decode()

def _call_bodhan(bgr, api_key, max_tokens=8192):
    payload = {
        "model": "indic-ocr",
        "messages": [{"role": "user", "content": [
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}},
        ]}],
        "max_tokens": max_tokens,
    }
    r = requests.post(BODHAN_URL,
                      headers={"Authorization": f"Bearer {api_key}"},
                      json=payload, timeout=(10, 300))
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"bodhan: {data['error'].get('message')}")
    data["_engine"] = "bodhan/indic-ocr"
    return data

_FALLBACK_OCR_PROMPT = (
    "You are an OCR engine. Transcribe this document page faithfully.\n"
    "Return ONLY JSON (no fences, no commentary) matching this schema:\n"
    "{\n"
    '  "blocks": [\n'
    "    {\n"
    '      "order": 1,\n'
    '      "type": "Text" | "Title" | "SectionHeader" | "List" | "Caption"\n'
    '              | "Table" | "Picture" | "PageHeader" | "PageFooter" | "Footnote",\n'
    '      "bbox_norm": [x0, y0, x1, y1] in [0,1] top-left, or null if uncertain,\n'
    '      "text": "exact content, preserve Latin+Devanagari+Telugu+... scripts"\n'
    "    }\n"
    "  ]\n"
    "}\n"
    "Rules: emit in natural reading order. Picture/signature/stamp blocks get text=\"\". "
    "Render tables as GitHub-style pipe tables. Never fabricate bbox coordinates — use null.\n"
    "\n"
    "Classify these as PageFooter / PageHeader (NOT Text) — they are page furniture, "
    "not body content, and must not be joined with the body across page breaks:\n"
    "  \u2022 scanner watermarks: \"Scanned by CamScanner\", \"Scanned with OKEN Scanner\"\n"
    "  \u2022 page codes / numbers: \"1-2-103\", \"Page 5 of 10\", \"[1]\"\n"
    "  \u2022 page-turn hints: \"[P.T.O.]\", \"Please Turn Over\", \"Continued on next page\"\n"
    "  \u2022 signatures or handwritten margin marks\n"
)

def _parse_json_object(text):
    s = text.strip()
    if s.startswith("```"):
        lines = s.splitlines()
        if lines[0].startswith("```"): lines = lines[1:]
        if lines and lines[-1].strip() == "```": lines = lines[:-1]
        s = "\n".join(lines)
    try: return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        return json.loads(m.group(0)) if m else {}

def _call_openrouter_ocr(bgr, api_key, model, timeout=180):
    H, W = bgr.shape[:2]
    payload = {
        "model": model, "temperature": 0,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": _FALLBACK_OCR_PROMPT},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}},
        ]}],
    }
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {api_key}",
                               "Content-Type": "application/json"},
                      json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"openrouter: {data['error']}")
    parsed = _parse_json_object(data["choices"][0]["message"]["content"])
    blocks = []
    for i, b in enumerate(parsed.get("blocks") or [], start=1):
        bn = b.get("bbox_norm")
        bbox_px = None
        if isinstance(bn, (list, tuple)) and len(bn) == 4:
            try:
                x0, y0, x1, y1 = [max(0.0, min(1.0, float(v))) for v in bn]
                bbox_px = [round(x0*W, 1), round(y0*H, 1), round(x1*W, 1), round(y1*H, 1)]
                if bbox_px[2] <= bbox_px[0] or bbox_px[3] <= bbox_px[1]: bbox_px = None
            except Exception: bbox_px = None
        blocks.append({"order": b.get("order", i), "type": b.get("type", "Text"),
                       "text": b.get("text", "") or "", "bbox_xyxy": bbox_px})
    return {"blocks": blocks, "_engine": f"openrouter/{model}"}

def _extract_bodhan_blocks(resp):
    if isinstance(resp.get("blocks"), list): return resp["blocks"]
    content = resp.get("choices", [{}])[0].get("message", {}).get("content")
    if isinstance(content, list): return content
    if isinstance(content, str):
        parsed = _parse_json_object(content)
        if isinstance(parsed, dict) and isinstance(parsed.get("blocks"), list):
            return parsed["blocks"]
        if isinstance(parsed, list): return parsed
    if isinstance(resp.get("data"), dict) and isinstance(resp["data"].get("blocks"), list):
        return resp["data"]["blocks"]
    return []

def ocr_with_fallback(bgr):
    try:
        resp = _call_bodhan(bgr, BODHAN_KEY)
        raw = _extract_bodhan_blocks(resp)
        normalised = []
        for i, b in enumerate(raw, start=1):
            if not isinstance(b, dict): continue
            normalised.append({"order": b.get("order", i),
                               "type":  b.get("type", "Text"),
                               "text":  b.get("text", "") or "",
                               "bbox_xyxy": b.get("bbox_xyxy") or b.get("bbox")})
        if not normalised or not any((b.get("text") or "").strip() for b in normalised):
            raise RuntimeError(f"Bodhan returned no usable blocks "
                               f"(got {len(normalised)}; top keys={list(resp.keys())[:6]})")
        return {"blocks": normalised, "_engine": "bodhan/indic-ocr"}
    except Exception as e:
        print(f"  Bodhan failed: {type(e).__name__}: {str(e)[:160]} \u2014 trying Gemini")
        if not OPENROUTER_KEY: raise RuntimeError("Bodhan failed and OPENROUTER_API_KEY is not set.")
        return _call_openrouter_ocr(bgr, OPENROUTER_KEY, OCR_FALLBACK_MODEL)


# ─────────────────────────────────────────────────────────────────────────
# A.2  Dedup  —  geometric + text subsumption (unchanged)
# ─────────────────────────────────────────────────────────────────────────
_PICTURE_TYPES = {"Picture", "Chart", "Diagram", "Image", "Figure", "picture"}

def _area(b): return max(0, b[2] - b[0]) * max(0, b[3] - b[1])
def _inter(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    return max(0, x1 - x0) * max(0, y1 - y0)
def _is_informative(block):
    if (block.get("text") or "").strip(): return True
    return block.get("type") in _PICTURE_TYPES
def _normalize_words(text):
    tbl = str.maketrans('', '', string.punctuation + "\u0964\u0965")
    return {w.translate(tbl).lower() for w in (text or "").split() if w.strip()}

def dedup_boxes(blocks, iou_threshold=0.3, contain_threshold=0.95, text_subsume_threshold=0.6):
    sorted_blocks = sorted(blocks,
        key=lambda b: (_is_informative(b), _area(b.get("bbox_xyxy") or [0,0,0,0])),
        reverse=True)
    kept, removed = [], []
    for c in sorted_blocks:
        if not c.get("bbox_xyxy"): kept.append(c); continue
        a = _area(c["bbox_xyxy"]) or 1.0
        c_words = _normalize_words(c.get("text") or "")
        drop_reason = None
        for k in kept:
            if not k.get("bbox_xyxy"): continue
            inter = _inter(c["bbox_xyxy"], k["bbox_xyxy"])
            if inter / a >= contain_threshold:
                drop_reason = f"contained in B{k.get('order')} ({inter/a:.0%})"; break
            ka = _area(k["bbox_xyxy"]) or 1.0
            iou = inter / (a + ka - inter)
            if iou >= iou_threshold:
                drop_reason = f"IoU {iou:.2f} with B{k.get('order')}"; break
            if inter > 0 and c_words:
                k_words = _normalize_words(k.get("text") or "")
                if k_words:
                    ov = len(c_words & k_words) / len(c_words)
                    if ov >= text_subsume_threshold:
                        drop_reason = f"text {ov:.0%} subsumed by B{k.get('order')}"
        if drop_reason: c = {**c, "_drop_reason": drop_reason}; removed.append(c)
        else: kept.append(c)
    kept.sort(key=lambda b: b.get("order", 0))
    return kept, removed


# ─────────────────────────────────────────────────────────────────────────
# A.3  Label  —  draw B0, B1, ... OUTSIDE each box (unchanged)
# ─────────────────────────────────────────────────────────────────────────
PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48),
           (145, 30, 180), (0, 128, 128), (240, 50, 230), (170, 110, 40)]

def draw_labeled(img_bgr, blocks):
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    try: font = ImageFont.truetype("arial.ttf", 24)
    except Exception: font = ImageFont.load_default()
    types = sorted({b.get("type", "?") for b in blocks})
    color_for = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(types)}
    W, H = pil.size
    for b in blocks:
        if not b.get("bbox_xyxy"): continue
        x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
        c = color_for[b.get("type", "?")]
        draw.rectangle([x0, y0, x1, y1], outline=c, width=3)
        tag = b["_label"]
        tw  = int(draw.textlength(tag, font=font)); th, pad = 28, 4
        tag_x = x0 - pad; tag_y = y0 - th - 2
        if tag_y < 0: tag_y = 0
        if tag_x < 0: tag_x = 0
        if tag_x + tw + 2*pad > W: tag_x = W - tw - 2*pad
        draw.rectangle([tag_x, tag_y, tag_x + tw + 2*pad, tag_y + th], fill=c)
        draw.text((tag_x + pad, tag_y + 2), tag, fill="white", font=font)
    return pil


# ─────────────────────────────────────────────────────────────────────────
# A.4  Ordering  —  Nemotron + Gemini fallback (REFINED PLAN_PROMPT)
# ─────────────────────────────────────────────────────────────────────────
#
# Changes vs previous version — STEP 4 (dedup) made MUCH more conservative.
# The old rule said "drop if ≥40% IoU AND text subset". Nemotron was still
# dropping option (\u0918) blocks that happened to share wording with earlier
# A/R-pattern questions. The new rule:
#   - raises IoU threshold to \u226550%,
#   - adds a THIRD condition (95%+ confidence it's a layout-model duplicate),
#   - gives explicit NEVER-DROP examples the model can match against.
#
# PLAN_PROMPT = (
#     "You are given ONE document page: the page image (every detected block is boxed "
#     "and labelled B0, B1, ...) and a JSON list of those blocks. Output the reading "
#     "order of the blocks.\n\n"
#     "JSON fields per block: id (the label on the image), type (title, section_header, "
#     "text, list_item, caption, footnote, table, picture, figure, formula, page_header, "
#     "page_footer, page_number), text (truncated to start+end for long blocks). Pictures "
#     "and tables may carry no text. Blocks are NOT given in reading order.\n\n"

#     "STEP 1 \u2014 LAYOUT. From the image, decide the layout: single column; two or more "
#     "columns; full-width blocks above/below/between columns; images or tables side by "
#     "side; a mix that changes down the page. Split into horizontal zones top-to-bottom "
#     "wherever a full-width block spans the page or the column structure changes.\n"
#     "- single column: top to bottom.\n"
#     "- 2 columns: read the LEFT column fully top-to-bottom, then the next column to its "
#     "right. NEVER read across a row of a multi-column zone.\n"
#     "- blocks side by side in one row (e.g. two images): left to right.\n\n"

#     "STEP 2 \u2014 BASE ORDER. Zones top to bottom; within a multi-column zone, left column "
#     "fully then right column fully. A caption stays with its figure/table (object then "
#     "caption). A formula stays where it sits. A title or section_header stays directly "
#     "before the text it introduces, in the same column.\n\n"

#     "STEP 3 \u2014 HEADERS, FOOTERS, PAGE NUMBERS. These stay at their VISUAL position in "
#     "the reading flow \u2014 page_header near the top, page_footer and page_number near the "
#     "bottom. Footnotes come after the body of the column they belong to. HEADERS AND "
#     "FOOTERS GO IN `order`, NEVER IN `dropped`.\n\n"

#     "STEP 4 \u2014 DEDUP (be VERY conservative).\n"
#     "Drop a block ONLY if ALL THREE of these are true:\n"
#     "  (a) its bbox overlaps another kept block's bbox by \u226550% IoU, AND\n"
#     "  (b) its text is a near-verbatim subset of the kept block's text \u2014 every word, "
#     "number, and label in the dropped block ALSO appears in the kept block, AND\n"
#     "  (c) you are at least 95% confident it is a layout-model duplicate (a coarse "
#     "box and a finer sub-box covering the SAME visual region).\n"
#     "If ANY of these is uncertain, KEEP the block.\n\n"

#     "The dedup rule exists ONLY to catch Bodhan returning both a coarse and a fine "
#     "box for the SAME region (e.g. B2 and B3 both wrap 'CHOICE BASED CREDIT SYSTEM' "
#     "and their bboxes mostly overlap). It does NOT exist to tidy up similar-looking "
#     "content elsewhere on the page.\n\n"

#     "Content you MUST ALWAYS KEEP (never list in `dropped`):\n"
#     "- Bilingual pairs: a Hindi/Telugu sentence and its English translation, even "
#     "when they sit next to each other.\n"
#     "- MCQ options with similar standard wording. For example question 14's option "
#     "(\u0918) and question 13's option (\u0918) both say \u0905\u092D\u093F\u0915\u0925\u0928 "
#     "(A) \u0914\u0930 \u0915\u093E\u0930\u0923 (R) \u0926\u094B\u0928\u094B\u0902 "
#     "\u0938\u0939\u0940 \u0939\u0948 \u2014 these belong to DIFFERENT questions and must "
#     "both be kept.\n"
#     "- Fill-in-blank dot sequences \u2014 the dots are unique content, not duplicates.\n"
#     "- Section headers that look like each other on different pages (e.g. 'PART-A' "
#     "appearing on multiple pages).\n"
#     "- Handwritten margin notes, numbering corrections, exam folios printed near a bbox.\n"
#     "- Headers and footers \u2014 they stay in `order` at their visual position (see STEP 3).\n\n"

#     "When in doubt \u2014 ESPECIALLY in doubt \u2014 put the block in `order`, not `dropped`.\n\n"

#     "STEP 5 \u2014 CROSS-PAGE CONTINUATION. If the first body text block of this page "
#     "begins mid-sentence (word fragment or lowercase), it continues from the previous "
#     "page \u2014 put it FIRST, before any figure/table that was above it. If the last body "
#     "text block ends cut off (mid-word or dangling), it continues to the next page \u2014 "
#     "it must be the very LAST item, so put headers/footers immediately before it. A "
#     "missing period alone is NOT a cut-off; on weak evidence, keep base order.\n\n"

#     "MCQ / Q&A pattern: a numbered question with sub-options ((\u0905), (\u0906), (a), (b) "
#     "\u2014 any Indic language) is one logical unit. Keep the question stem and ALL its "
#     "options together in order. Options for the same question are NEVER duplicates of "
#     "each other.\n\n"

#     "Every id in `order` + `dropped` together must appear exactly once across the two "
#     "lists, and only ids from the input list.\n\n"

#     'Return ONLY this JSON, no prose, no code fences:\n'
#     '{"layout": "<brief layout description>", '
#     '"order": [...ids in reading order], '
#     '"dropped": [...ids whose content is TRULY duplicated by another block]}\n\n'

#     "Regions:\n"
# )

PLAN_PROMPT = (
    "You are given ONE document page: the page image (every detected block is boxed "
    "and labelled B0, B1, ...) and a JSON list of those blocks. Output the reading "
    "order of these blocks.\n\n"

    "CORE PRINCIPLE \u2014 VISUAL POSITION BEATS TYPE LABELS.\n"
    "Reading order = the order a human reads the page out loud, scanning the way the "
    "page's script runs (top-to-bottom, left-to-right for Latin / Devanagari / "
    "Telugu / Tamil / Kannada / Bengali / Gujarati / Odia; right-to-left for Arabic / "
    "Urdu). Follow the VISUAL layout. The `type` field is an upstream guess \u2014 trust "
    "what you SEE in the image when it conflicts with the type. Use the `bbox` field "
    "([x0, y0, x1, y1] normalised to [0,1]) to resolve position when the image is "
    "ambiguous.\n"
    "Do NOT batch blocks by type (do NOT put all headings first, all footers last, "
    "all metadata grouped). Follow actual position.\n\n"

    "JSON fields per block: id, type (title | section_header | text | list_item | "
    "caption | footnote | table | picture | figure | formula | page_header | "
    "page_footer | page_number), bbox [x0,y0,x1,y1] normalised, text (preview). "
    "Blocks are NOT in reading order.\n\n"

    "STEP 1 \u2014 ZONES.\n"
    "Split the page into horizontal zones top-to-bottom. A new zone starts where "
    "column structure changes, or where a full-width block separates two text groups "
    "of a column.\n\n"

    "STEP 2 \u2014 ORDER WITHIN A ZONE.\n"
    "- Single-column zone: top to bottom.\n"
    "- Multi-column zone: read each column fully top-to-bottom, in the direction of "
    "the script (left-to-right columns for Indic/Latin). NEVER read across columns.\n"
    "- Within a single-column flow, when ONE row holds two or more side-by-side "
    "blocks (e.g. a left-aligned label and a right-aligned label on the same y-"
    "coordinate), read that row's blocks in script direction, THEN continue down.\n\n"

    "STEP 3 \u2014 LABELLING BLOCKS GO BEFORE WHAT THEY LABEL.\n"
    "A heading, section marker, title-above-content, or any block whose role is to "
    "INTRODUCE the next block of content, goes IMMEDIATELY BEFORE that content. "
    "If multiple such labelling blocks exist on the page, EACH stays before ITS OWN "
    "content \u2014 do NOT place them all at the top, do NOT place them all at the end.\n"
    "A caption that sits BELOW a figure/table stays with it (figure/table then caption). "
    "A formula stays in its position.\n\n"

    "STEP 4 \u2014 TRUE PAGE FURNITURE vs BODY CONTENT.\n"
    "True page furniture = small text at the very top or bottom edges of the page "
    "that would repeat on other pages of the same document: running header, page "
    "number, folio code at a corner, 'Page N of M', scanner watermark, continuation "
    "hints like '[P.T.O.]' or 'Continued on next page'. These stay at their visual "
    "position (top or bottom). They go in `order`, NEVER in `dropped`.\n\n"
    "BODY CONTENT (keep in reading flow, do NOT treat as furniture even if the "
    "upstream type says `page_header` or `page_footer`):\n"
    "- Metadata lines that sit with the document's title or masthead (dates, "
    "reference numbers, totals, prices, author names, issue numbers, duration or "
    "limits of any kind).\n"
    "- Section labels, part markers, sub-headings positioned in the body.\n"
    "- Standalone short labels (addressee, date in a letter, a field name).\n"
    "The test: would this text be IDENTICAL on every page of this document? If no, "
    "it is body content, not furniture.\n\n"

    "STEP 5 \u2014 DEDUP (very conservative).\n"
    "Drop a block ONLY if ALL THREE hold:\n"
    "  (a) its bbox overlaps another kept block's bbox by \u226550% IoU, AND\n"
    "  (b) its text is a near-verbatim subset of that kept block's text, AND\n"
    "  (c) you are 95%+ confident this is an upstream duplicate (coarse box + fine "
    "sub-box over the SAME region).\n"
    "If ANY is uncertain \u2192 KEEP.\n\n"
    "NEVER drop:\n"
    "- Short standalone labels anywhere on the page.\n"
    "- Bilingual pairs (any Indic language + its English translation).\n"
    "- Options or sub-items of different numbered items that share standard wording.\n"
    "- Fill-in-blank dot or dash sequences.\n"
    "- Repeated section labels on different pages.\n"
    "- Any header, footer, or page number.\n"
    "When in doubt \u2192 `order`, not `dropped`.\n\n"

    "STEP 6 \u2014 CROSS-PAGE CONTINUATION.\n"
    "First body block begins mid-sentence (lowercase or word fragment) \u2192 continues "
    "from previous page, put FIRST.\n"
    "Last body block ends mid-word or dangling \u2192 continues to next page, put LAST "
    "with headers/footers immediately before it.\n"
    "Missing period alone is NOT a cut-off. On weak evidence, keep base order.\n"
    "A numbered item with sub-options (any script) is one unit \u2014 keep stem and all "
    "options together.\n\n"

    "Two short examples of the ordering principle. These are PATTERNS to generalise "
    "from, not templates for a specific document.\n\n"
    "Example 1 (Devanagari textbook page): [page code at a corner] \u2192 [chapter title "
    "centred] \u2192 [author line] \u2192 [section heading 1] \u2192 [body of section 1] \u2192 "
    "[section heading 2] \u2192 [body of section 2] \u2192 [figure] \u2192 [figure caption] \u2192 "
    "[running footer with page number].\n"
    "Example 2 (Tamil form page): [form title] \u2192 [left label] [right label on same "
    "row] \u2192 [section A heading] \u2192 [instructions] \u2192 [fields of section A] \u2192 "
    "[section B heading] \u2192 [instructions] \u2192 [fields of section B] \u2192 [bottom-right "
    "continuation hint].\n\n"
    "In both: headings come BEFORE their content; side-by-side blocks read in script "
    "direction on their row; true edge furniture stays at edges; nothing is batched "
    "by type.\n\n"

    "STEP 7 \u2014 SELF-CHECK (do this BEFORE emitting the output).\n"
    "Scan your proposed `order` and check these three things:\n"
    "  (1) For every block whose text visually labels something (a heading, section "
    "marker, title): is the block IMMEDIATELY after it in your `order` a block that "
    "sits DIRECTLY BELOW it on the page (not some far-away block)? If a heading in "
    "your order is followed by something that is geometrically far from it, you have "
    "grouped by type \u2014 fix it.\n"
    "  (2) Look at the first 3 ids in your `order` and the last 3. Are the first 3 "
    "the ones with the SMALLEST y-coordinate on the page, and the last 3 the ones "
    "with the LARGEST? If not, your order is not following visual top-to-bottom flow "
    "\u2014 fix it.\n"
    "  (3) Did you include EVERY input id exactly once across `order` + `dropped`? "
    "If any id is missing, add it to `order` at its visual position.\n"
    "Only emit the output after these three checks pass.\n\n"

    'Return ONLY this JSON, no prose, no code fences, no explanation:\n'
    '{"layout": "<brief description of layout>", '
    '"order": [...ids in reading order], '
    '"dropped": [...ids that are TRUE upstream duplicates]}\n\n'
    "Regions:\n"
)

def _build_order_payload(blocks, W, H, text_cap=200):
    out = []
    for b in blocks:
        bb = b.get("bbox_xyxy")
        entry = {"id": b["_label"], "type": b.get("type", "text"),
                 "bbox": [round(bb[0]/W, 3), round(bb[1]/H, 3),
                          round(bb[2]/W, 3), round(bb[3]/H, 3)] if bb else None}
        t = (b.get("text") or "").strip()
        if t:
            entry["text"] = t if len(t) <= text_cap else t[:text_cap//2] + " [\u2026] " + t[-text_cap//2:]
        out.append(entry)
    return out

def _call_ordering_model(labeled_pil, payload, api_key, model, timeout=180):
    buf = io.BytesIO(); labeled_pil.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    body = {"model": model, "temperature": 0,
            "messages": [{"role": "user", "content": [
                {"type": "text", "text": PLAN_PROMPT + json.dumps(payload, ensure_ascii=False)},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}]}
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {api_key}",
                               "Content-Type": "application/json"},
                      json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
    return _parse_json_object(data["choices"][0]["message"]["content"])

def order_with_fallback(labeled_pil, payload):
    known = {e["id"] for e in payload}
    last_err = None
    for model in ORDER_MODELS:
        try:
            print(f"    trying {model} \u2026")
            j = _call_ordering_model(labeled_pil, payload, OPENROUTER_KEY, model)
            order   = [i for i in (j.get("order")   or []) if i in known]
            dropped = [i for i in (j.get("dropped") or []) if i in known]
            seen = set(order) | set(dropped)
            order += [i for i in known if i not in seen]
            print(f"    \u2190 layout: {j.get('layout','?')!r}  order={len(order)}  dropped={len(dropped)}")
            return {"order": order, "dropped": dropped, "layout": j.get("layout"), "_model": model}
        except Exception as e:
            print(f"    {model} failed: {type(e).__name__}: {str(e)[:160]}")
            last_err = e
    raise RuntimeError(f"all ordering models failed: {last_err}")


# ─────────────────────────────────────────────────────────────────────────
# A.5  Run per-page: preprocess \u2192 OCR \u2192 dedup \u2192 order
# ─────────────────────────────────────────────────────────────────────────
BODHAN_TO_STD = {
    "Title":"title", "SectionHeader":"heading", "Text":"paragraph",
    "List":"list_item", "Caption":"caption", "Table":"paragraph",
    "Picture":"picture", "Chart":"picture", "Diagram":"picture",
    "Image":"picture", "Figure":"picture",
    "PageHeader":"header", "PageFooter":"footer", "PageNumber":"footer",
    "Header":"header", "Footer":"footer", "Footnote":"paragraph",
    "Infobox":"paragraph", "Folio":"footer",
}

ocr_responses_A, plans_A, preprocessed_A = {}, {}, {}
ocr_failures_A, order_failures_A = {}, {}

for pno in page_indices:
    print(f"\n\u2500\u2500 A \u00b7 page {pno+1}/{total} \u2500\u2500")
    set_stage("A", "preprocess")
    img = preprocess_page(pno)
    preprocessed_A[pno] = img

    set_stage("A", "ocr")
    try:
        ocr_responses_A[pno] = ocr_with_fallback(img)
        print(f"  ocr ok: {ocr_responses_A[pno]['_engine']}  "
              f"({len(ocr_responses_A[pno]['blocks'])} blocks)")
        print(f"  \u2500\u2500 OCR raw blocks (page {pno+1}) \u2500\u2500")
        for _bi, _b in enumerate(ocr_responses_A[pno]["blocks"]):
            _show_block("[OCR]", _b, label=f"#{_bi}")
    except Exception as e:
        ocr_failures_A[pno] = str(e); print(f"  OCR failed: {e}"); continue

    set_stage("A", "dedup+order")
    kept, removed = dedup_boxes(ocr_responses_A[pno]["blocks"])
    for i, b in enumerate(kept): b["_label"] = f"B{i}"
    print(f"  dedup: {len(ocr_responses_A[pno]['blocks'])} \u2192 kept {len(kept)}, removed {len(removed)}")
    print(f"  \u2500\u2500 DEDUP kept (page {pno+1}) \u2500\u2500")
    for _b in kept:
        _show_block("[KEPT]", _b)
    if removed:
        print(f"  \u2500\u2500 DEDUP removed (page {pno+1}) \u2500\u2500")
        for _b in removed:
            reason = _b.get("_drop_reason", "?")
            print(f"    [DEDUP-DROP] #{_b.get('order','?')} reason={reason}  text={_preview(_b.get('text'))!r}")

    labeled = draw_labeled(img, kept)
    labeled.save(PAGE_DIR_A / f"{stem}_p{pno+1}_labeled.png")

    H, W = img.shape[:2]
    payload = _build_order_payload(kept, W, H)
    try:
        plan = order_with_fallback(labeled, payload)
    except Exception as e:
        order_failures_A[pno] = str(e)
        plan = {"order": [b["_label"] for b in kept], "dropped": [],
                "layout": "fallback (ordering failed)", "_model": "fallback"}
    plans_A[pno] = {"kept": kept, **plan}

    # ── ORDER-PHASE DIAGNOSTICS ───────────────────────────────────────────
    _idmap = {b["_label"]: b for b in kept}
    _known = set(_idmap.keys())
    _in_order   = [r for r in plan.get("order", [])   if r in _known]
    _in_dropped = [r for r in plan.get("dropped", []) if r in _known]
    _omitted    = sorted(_known - set(_in_order) - set(_in_dropped))

    print(f"  \u2500\u2500 QWEN order (page {pno+1}, model={plan.get('_model','?')}) \u2500\u2500")
    for pos, rid in enumerate(_in_order, 1):
        b = _idmap[rid]
        print(f"    [ORDER {pos:>2}] {rid:>4}  type={b.get('type','?'):<14} "
              f"text={_preview(b.get('text'))!r}")
    if _in_dropped:
        print(f"  \u2500\u2500 QWEN dropped (its own `dropped` list) \u2500\u2500")
        for rid in _in_dropped:
            b = _idmap[rid]
            print(f"    [QWEN-DROP] {rid:>4} type={b.get('type','?'):<14} "
                  f"text={_preview(b.get('text'))!r}")
    if _omitted:
        print(f"  \u26a0 QWEN OMITTED (not in order, not in dropped) \u2014 "
              f"these would be re-appended by the safety-net:")
        for rid in _omitted:
            b = _idmap[rid]
            print(f"    [OMITTED]   {rid:>4} type={b.get('type','?'):<14} "
                  f"text={_preview(b.get('text'))!r}")

# ─────────────────────────────────────────────────────────────────────────
# A.6  Build page_dicts with BLOCK IDs (block_id, bodhan_type, bbox_xyxy)
#      These travel through the pipeline into Qwen and the final sidecar.
# ─────────────────────────────────────────────────────────────────────────
set_stage("A", "build")
page_dicts_A = []
for pno in page_indices:
    if pno not in plans_A: continue
    plan = plans_A[pno]
    idmap = {b["_label"]: b for b in plan["kept"]}
    pil = Image.fromarray(cv2.cvtColor(preprocessed_A[pno], cv2.COLOR_BGR2RGB))
    regions = []
    for i, rid in enumerate(plan["order"], start=1):
        if rid not in idmap: continue
        b = idmap[rid]
        std = BODHAN_TO_STD.get(b.get("type"), "paragraph")

        # Stable block id survives through Qwen formatter and into the sidecar.
        block_id = f"p{pno+1}_{rid}"

        reg = {
            "id":          f"A_p{pno+1}_r{i}",
            "block_id":    block_id,
            "page":        pno + 1,
            "type":        std,
            "bodhan_type": b.get("type", "Text"),
            "bbox_xyxy":   b.get("bbox_xyxy"),
        }
        if std == "picture" and b.get("bbox_xyxy"):
            x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
            crop_path = CROP_DIR_A / f"{stem}_p{pno+1}_{rid}.png"
            crop_path.parent.mkdir(parents=True, exist_ok=True)
            pil.crop((x0, y0, x1, y1)).save(crop_path)
            reg["image_path"] = str(crop_path)
        else:
            reg["text"] = b.get("text") or ""
        regions.append(reg)
    page_dicts_A.append({"page": pno + 1, "regions": regions})

# Final page_dicts summary — what will be fed to build_docx
print(f"\n\u2500\u2500 FINAL page_dicts_A \u2500\u2500")
for _pd in page_dicts_A:
    print(f"\n  page {_pd['page']}: {len(_pd['regions'])} region(s)")
    for _reg in _pd["regions"]:
        _rid    = _reg["id"]
        _bid    = _reg.get("block_id", "?")
        # _rtype  = _reg.get("type", "paragraph")
        _rtype  = _reg.get("bodhan_type", "paragraph")
        _bodh   = _reg.get("bodhan_type", "?")
        _bbox   = _reg.get("bbox_xyxy")
        _bbox_s = "null" if _bbox is None else f"[{_bbox[0]:.0f},{_bbox[1]:.0f},{_bbox[2]:.0f},{_bbox[3]:.0f}]"
        if "image_path" in _reg:
            print(f"    {_rid:<24} block={_bid:<14} type={_rtype:<10}  "
                  f"PICTURE path={_reg['image_path']}")
        else:
            print(f"    {_rid:<24} block={_bid:<14} type={_rtype:<10} "
                  f"bodhan={_bodh:<13} bbox={_bbox_s:<25}  "
                  f"text={_preview(_reg.get('text'))!r}")

cross_A = cross_page(page_dicts_A)
print(f"\nA \u00b7 page_dicts: {len(page_dicts_A)}  \u00b7  cross-page joins: {len(cross_A)}")

docx_A_path = METHOD_A_DIR / f"{stem}__A.docx"
final_A = save_docx_with_fallback(page_dicts_A, cross_A, docx_A_path)
print(f"A \u00b7 saved {final_A}")
set_stage("-", "-")


[metrics] cleared

── A · page 1/19 ──
  ocr ok: bodhan/indic-ocr  (7 blocks)
  ── OCR raw blocks (page 1) ──
    [OCR]     #0  type=PageHeader      bbox=[308,187,1320,611]         text=''
    [OCR]     #1  type=SectionHeader   bbox=[352,213,1283,371]         text='KALPATARU FIRST GRADE SCIENCE COLLEGE, TIPTUR ⏎ QUALITY INDICATOR FRAMEW'
    [OCR]     #2  type=SectionHeader   bbox=[450,430,1187,602]         text='CRITERION - 2 ⏎ 2.5.1 – TEST QUESTION PAPERS'
    [OCR]     #3  type=Text            bbox=[341,685,1357,780]         text='File contains Sample Test Question Papers for Internal Assessment'
    [OCR]     #4  type=Table           bbox=[336,880,1368,1564]        text='<table> ⏎ <thead> ⏎ <tr> ⏎ <th>Sl.<br/>No</th> ⏎ <th>SUBJECT</th> ⏎ <th>PAGE NO.'
    [OCR]     #5  type=Text            bbox=[1186,2260,1632,2309]      text='Scanned with OKEN Scanner'
    [OCR]     #6  type=PageFooter      bbox=[1098,2252,1638,2314]      text=''
  dedup: 7 → kept 5, removed 2
  ── DEDUP kept (pag

## Method B — Gemini chunk-wise → DOCX

Chunk-wise pipeline, new design:

1. **Chunk** — pages split into groups of `CHUNK_SIZE` (default 3)
2. **One Gemini call per chunk** — all N page-images in one request. The prompt asks for a strict JSON output: typed blocks per page (title / heading / paragraph / list_item / table / picture_description) plus `chunk_starts_continuation` / `chunk_ends_continuation` flags
3. **Within a chunk**: Gemini handles cross-page continuations naturally (sentences, lists, tables that span pages inside the chunk flow as one)
4. **Between chunks**: Python stitches using the continuation flags; if chunk K ended mid-sentence AND chunk K+1 starts mid-sentence, merge last block of K with first block of K+1
5. **DOCX** — same shared builder as Method A

No separate OCR step, no bbox labelling, no dedup — Gemini does it all in one pass per chunk.


In [68]:
# # ─── CELL 8: Method B ─────────────────────────────────────────────────────
# # NOTE: do NOT call reset_metrics() here — we want A's metrics kept alongside B's.

# METHOD_B_PROMPT_HEAD = """You are a meticulous document transcriber. You are given {n} consecutive pages ({first}–{last}) of ONE scanned document, as images in order.

# Transcribe ALL content on these pages in reading order, into a structured JSON format described below. The content may be in English, Hindi, Telugu, Kannada, or any Indic script — preserve every script EXACTLY as printed.

# FIDELITY — highest priority
# - Transcribe EXACTLY what is printed. Never translate, summarize, "correct" spelling, normalize punctuation, merge or reorder questions.
# - Preserve multilingual text (UTF-8), the document's own numbering ("1.", "(a)", "(क)", "प्रश्न 1"), marks notation ("[5 marks]", "3×2=6"), times, "OR" / "अथवा" lines, and fill-in-blank dot sequences (as literal dots).
# - Text you cannot read: write "[illegible]". Never guess.

# PAGE FURNITURE — OMIT entirely (no text, no placeholder)
# - Scanner watermarks ("Scanned with OKEN Scanner", "Scanned by CamScanner")
# - Page numbers, "[P.T.O.]", "Continued on next page", exam folios like "1-2-103"
# - Repeating running headers/footers (college name shown ONLY on the first page is NOT furniture; the same text on EVERY page is)

# READING ORDER
# - Multi-column: read the LEFT column fully top-to-bottom, then the next column.
# - A caption stays with its figure/table.
# - A heading stays directly before the text it introduces.

# CROSS-PAGE CONTINUATIONS (within this chunk)
# - If a sentence, list, or table spans two pages inside this chunk, produce it as ONE natural flow — do NOT restart, do NOT label the page break. The reader should see the complete sentence/list/table.
# - Set `chunk_starts_continuation` = true ONLY if the FIRST block of page {first} obviously starts mid-sentence (lowercase word, fragment) from a page not shown.
# - Set `chunk_ends_continuation` = true ONLY if the LAST block of page {last} is clearly cut off mid-sentence and continues on a page not shown.

# PICTURES
# - You cannot crop images. For diagrams, charts, stamps, signatures: use type "picture_description" with a short factual description (what it depicts, visible labels). Do not transcribe stamps or signatures as text.

# OUTPUT — strict JSON, no prose, no code fences. Schema:

# {{
#   "pages": [
#     {{
#       "page_num": <int, matches the printed page number if visible, else the ordinal>,
#       "blocks": [
#         {{"type": "title", "text": "..."}},
#         {{"type": "heading", "level": 1|2|3, "text": "..."}},
#         {{"type": "paragraph", "text": "..."}},
#         {{"type": "list_item", "text": "..."}},
#         {{"type": "table", "rows": [["cell", "cell"], ["cell", "cell"]]}},
#         {{"type": "picture_description", "text": "..."}}
#       ]
#     }}
#   ],
#   "chunk_starts_continuation": false,
#   "chunk_ends_continuation": false
# }}

# Return the JSON only.
# """

# def _pil_data_uri(bgr):
#     return f"data:image/png;base64,{_b64_png(bgr)}"

# def _call_B_chunk(pnos, images_bgr, model, timeout=420):
#     """One Gemini call per chunk. Returns parsed dict."""
#     first, last, n = pnos[0] + 1, pnos[-1] + 1, len(pnos)
#     text = METHOD_B_PROMPT_HEAD.format(n=n, first=first, last=last)
#     content = [{"type": "text", "text": text}]
#     for img in images_bgr:
#         content.append({"type": "image_url", "image_url": {"url": _pil_data_uri(img)}})
#     body = {"model": model, "temperature": 0,
#             "messages": [{"role": "user", "content": content}]}
#     r = requests.post(OPENROUTER_URL,
#                       headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
#                                "Content-Type": "application/json"},
#                       json=body, timeout=timeout)
#     r.raise_for_status()
#     data = r.json()
#     if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
#     parsed = _parse_json_object(data["choices"][0]["message"]["content"])
#     if not isinstance(parsed.get("pages"), list):
#         raise ValueError(f"bad JSON shape (no 'pages' list): {str(parsed)[:200]}")
#     return parsed

# def _fallback_B_page(pno):
#     """When the chunk LLM fails on a single page even after splitting."""
#     return {"page_num": pno + 1,
#             "blocks": [{"type": "paragraph", "text": f"[Method B could not transcribe page {pno+1}]"}]}

# def solve_B_chunk(pnos, images_bgr):
#     """Try the chunk; on failure, split in half; down to single pages, then fallback."""
#     try:
#         print(f"  [B] Gemini on pages {pnos[0]+1}–{pnos[-1]+1} …")
#         r = _call_B_chunk(pnos, images_bgr, METHOD_B_MODEL)
#         print(f"  ← pages={len(r.get('pages', []))}  cont s/e: "
#               f"{r.get('chunk_starts_continuation', False)}/{r.get('chunk_ends_continuation', False)}")
#         return r, {}
#     except Exception as e:
#         print(f"  B chunk failed: {type(e).__name__}: {str(e)[:180]}")
#         if len(pnos) == 1:
#             return {"pages": [_fallback_B_page(pnos[0])],
#                     "chunk_starts_continuation": False,
#                     "chunk_ends_continuation": False}, {pnos[0]: str(e)}
#         mid = len(pnos) // 2
#         print(f"  ⚠ splitting chunk {pnos[0]+1}–{pnos[-1]+1}")
#         left,  f_l = solve_B_chunk(pnos[:mid], images_bgr[:mid])
#         right, f_r = solve_B_chunk(pnos[mid:], images_bgr[mid:])
#         merged = {"pages": left["pages"] + right["pages"],
#                   "chunk_starts_continuation": left["chunk_starts_continuation"],
#                   "chunk_ends_continuation":   right["chunk_ends_continuation"]}
#         return merged, {**f_l, **f_r}


# # ─────────────────────────────────────────────────────────────────────────
# # B.1  Preprocess each page once (same CLAHE as Method A — fair comparison)
# # ─────────────────────────────────────────────────────────────────────────
# set_stage("B", "preprocess")
# preprocessed_B = {}
# for pno in page_indices:
#     # If Method A already preprocessed this page, reuse it (same image).
#     img_cached = preprocessed_A.get(pno)
#     preprocessed_B[pno] = img_cached if img_cached is not None else preprocess_page(pno)


# # ─────────────────────────────────────────────────────────────────────────
# # B.2  Chunk pages, call Gemini per chunk
# # ─────────────────────────────────────────────────────────────────────────
# chunks_B = [page_indices[i:i + CHUNK_SIZE] for i in range(0, len(page_indices), CHUNK_SIZE)]
# print(f"\nB · {len(page_indices)} page(s) → {len(chunks_B)} chunk(s) of up to {CHUNK_SIZE}")

# set_stage("B", "chunked_llm")
# chunk_results_B, chunk_failures_B = [], {}
# for ci, chunk in enumerate(chunks_B, 1):
#     print(f"\n── B · chunk {ci}/{len(chunks_B)}: pages {chunk[0]+1}–{chunk[-1]+1} ──")
#     images = [preprocessed_B[p] for p in chunk]
#     result, fails = solve_B_chunk(chunk, images)
#     chunk_results_B.append({"pnos": chunk, **result})
#     chunk_failures_B.update(fails)


# # ─────────────────────────────────────────────────────────────────────────
# # B.3  Flatten chunks → page_dicts, handle BETWEEN-chunk continuation
# # ─────────────────────────────────────────────────────────────────────────
# set_stage("B", "build")
# page_dicts_B = []
# cross_B = []
# prev_chunk_last_rid = None
# prev_chunk_ends_cont = False

# for ci, result in enumerate(chunk_results_B):
#     chunk_pnos = result["pnos"]
#     pages = result.get("pages") or []
#     # Pad / trim to the chunk size if the model under/over-returned
#     for local_i, pno in enumerate(chunk_pnos):
#         if local_i < len(pages):
#             page_blocks = pages[local_i].get("blocks") or []
#         else:
#             page_blocks = [{"type": "paragraph", "text": f"[page {pno+1}: no content returned]"}]
#         regions = []
#         for bi, b in enumerate(page_blocks, start=1):
#             if not isinstance(b, dict): continue
#             btype = b.get("type", "paragraph")
#             reg = {"id": f"B_c{ci+1}_p{pno+1}_r{bi}", "type": btype}
#             if btype == "table":
#                 reg["rows"] = b.get("rows") or []
#             else:
#                 reg["text"] = b.get("text") or ""
#                 if btype == "heading":
#                     reg["level"] = int(b.get("level", 1) or 1)
#             regions.append(reg)
#         page_dicts_B.append({"page": pno + 1, "regions": regions})

#     # BETWEEN-chunk continuation: if the previous chunk ENDED mid-sentence AND
#     # THIS chunk STARTS mid-sentence, merge the last body of previous into first body of this.
#     if ci > 0 and prev_chunk_ends_cont and result.get("chunk_starts_continuation") and prev_chunk_last_rid:
#         # find the first real-body region of this chunk's pages
#         for pd in page_dicts_B[-len(chunk_pnos):]:
#             found = False
#             for r in pd["regions"]:
#                 if _is_real_body({"type": r["type"], "text": r.get("text", "")}):
#                     cross_B.append((prev_chunk_last_rid, r["id"]))
#                     found = True; break
#             if found: break

#     # remember the last body region of THIS chunk for the next iteration
#     prev_chunk_ends_cont = bool(result.get("chunk_ends_continuation"))
#     last_rid = None
#     for pd in reversed(page_dicts_B[-len(chunk_pnos):]):
#         for r in reversed(pd["regions"]):
#             if _is_real_body({"type": r["type"], "text": r.get("text", "")}):
#                 last_rid = r["id"]; break
#         if last_rid: break
#     prev_chunk_last_rid = last_rid

# # ALSO run Python's cross-page detector WITHIN chunks for pages where the model
# # didn't flow a continuation naturally (safety net). Only adds joins; never undoes any.
# extra = cross_page(page_dicts_B)
# # merge, dedup
# existing = {j for j in cross_B}
# for j in extra:
#     if j not in existing: cross_B.append(j)

# print(f"\nB · page_dicts: {len(page_dicts_B)}  ·  cross-page joins: {len(cross_B)}  "
#       f"({sum(1 for _ in chunk_results_B if _.get('chunk_ends_continuation'))} chunks ended mid-sentence)")

# docx_B_path = METHOD_B_DIR / f"{stem}__B.docx"
# final_B = save_docx_with_fallback(page_dicts_B, cross_B, docx_B_path)
# print(f"B · saved {final_B}")
# set_stage("-", "-")


## Comparison — tokens · latency · cost · outcome

Breaks down the metrics logged during both runs:

- **Per stage, per engine**: calls, tokens in/out, API time, USD cost
- **Per method total**: wall-clock, cost per page, blended $/1M tokens
- **Outcome**: file size, paragraph count, image count, table count

The OCR costs should be roughly equal between methods if preprocessing is 
the same. The real signal is in `dedup+order` (A) vs `chunked_llm` (B) — 
same job, different strategy.


In [69]:
# # ─── CELL 10: comparison report ──────────────────────────────────────────
# _NAMES = {
#     "A": "A · Bodhan OCR (+ Gemini fallback) → Nemotron ordering → DOCX",
#     "B": f"B · Gemini chunk-wise ({CHUNK_SIZE} pages/call) → DOCX",
# }

# def _stage_order(method):
#     seen = []
#     for c in CALLS:
#         if c["method"] == method and c["stage"] not in seen:
#             seen.append(c["stage"])
#     return seen

# def method_report(method):
#     calls = [c for c in CALLS if c["method"] == method and c["engine"] != "_wallclock_"]
#     wall  = [c for c in CALLS if c["method"] == method and c["engine"] == "_wallclock_"]
#     if not calls and not wall:
#         print(f"\n[{method}] no calls logged"); return None

#     print(f"\n{'═'*98}")
#     print(f" METHOD {_NAMES[method]}")
#     print(f"{'═'*98}")
#     print(f" {'stage':<16}{'engine':<32}{'calls ok/fail':>14}{'tok in':>10}{'tok out':>10}"
#           f"{'api s':>8}{'cost $':>10}")
#     print(" " + "─"*96)

#     tot = dict(ok=0, fail=0, i=0, o=0, s=0.0, cost=0.0, unpriced=0)
#     for st in _stage_order(method):
#         engines = sorted({c["engine"] for c in calls if c["stage"] == st})
#         for en in engines:
#             cs = [c for c in calls if c["stage"] == st and c["engine"] == en]
#             ok = [c for c in cs if c["ok"]]
#             cost = sum((c["cost"] or 0) for c in ok)
#             unp  = sum(1 for c in ok if c["cost"] is None)
#             vals = (len(ok), len(cs) - len(ok), sum(c["tok_in"] for c in ok),
#                     sum(c["tok_out"] for c in ok),
#                     sum(c["seconds"] for c in cs), cost)
#             flag = " *" if unp else ""
#             print(f" {st[:16]:<16}{en[:31]:<32}{f'{vals[0]}/{vals[1]}':>14}"
#                   f"{vals[2]:>10,}{vals[3]:>10,}{vals[4]:>8.1f}{vals[5]:>10.4f}{flag}")
#             tot["ok"] += vals[0]; tot["fail"] += vals[1]
#             tot["i"]  += vals[2]; tot["o"] += vals[3]
#             tot["s"]  += vals[4]; tot["cost"] += vals[5]
#             tot["unpriced"] += unp

#     wall_by_stage = defaultdict(float)
#     for c in wall:
#         wall_by_stage[c["stage"]] += c["seconds"]
#     print(" " + "─"*96)
#     print(" wall-clock by stage (s): " +
#           " · ".join(f"{k} {v:.1f}" for k, v in wall_by_stage.items()))
#     total_wall = sum(wall_by_stage.values())
#     n_pages = len(page_indices)
#     per_page_cost = tot["cost"] / max(n_pages, 1)
#     print(f" TOTAL  wall {total_wall:.1f}s ({total_wall/max(n_pages,1):.1f}s/page)  ·  "
#           f"API {tot['s']:.1f}s  ·  tokens in {tot['i']:,} / out {tot['o']:,}  ·  "
#           f"cost ${tot['cost']:.4f} (${per_page_cost:.4f}/page)")
#     if tot["unpriced"]:
#         print(f" *  {tot['unpriced']} call(s) unpriced (Bodhan). Set BODHAN_USD_PER_PAGE in Cell 1.")
#     return {"wall_s": total_wall, "api_s": tot["s"],
#             "tok_in": tot["i"], "tok_out": tot["o"],
#             "cost": tot["cost"], "unpriced": tot["unpriced"]}

# def file_outcome(method, path):
#     if not Path(path).exists():
#         return {"path": None, "size_kb": 0, "paragraphs": 0, "tables": 0, "pictures": 0}
#     d = Document(path)
#     paras   = sum(1 for p in d.paragraphs if (p.text or "").strip())
#     tables  = len(d.tables)
#     pictures = 0
#     for sh in d.inline_shapes:
#         try: pictures += 1
#         except Exception: pass
#     return {"path": str(path),
#             "size_kb": round(Path(path).stat().st_size / 1024, 1),
#             "paragraphs": paras, "tables": tables, "pictures": pictures}

# # per-method breakdowns
# tot_A = method_report("A")
# tot_B = method_report("B")

# out_A = file_outcome("A", final_A if 'final_A' in dir() else "")
# out_B = file_outcome("B", final_B if 'final_B' in dir() else "")

# print(f"\n{'═'*98}")
# print(f" SIDE-BY-SIDE  (pages processed: {len(page_indices)})")
# print(f"{'═'*98}")
# print(f" {'metric':<28}{'Method A':>30}{'Method B':>30}")
# print(" " + "─"*88)
# def _row(label, a, b, fmt=str):
#     print(f" {label:<28}{fmt(a):>30}{fmt(b):>30}")

# if tot_A and tot_B:
#     _row("wall-clock (s)",   tot_A["wall_s"], tot_B["wall_s"], lambda x: f"{x:.1f}")
#     _row("API time (s)",     tot_A["api_s"],  tot_B["api_s"],  lambda x: f"{x:.1f}")
#     _row("tokens in",        tot_A["tok_in"], tot_B["tok_in"], lambda x: f"{x:,}")
#     _row("tokens out",       tot_A["tok_out"],tot_B["tok_out"],lambda x: f"{x:,}")
#     _row("cost (USD)",       tot_A["cost"],   tot_B["cost"],   lambda x: f"${x:.4f}")
#     _row("cost per page",    tot_A["cost"] / len(page_indices),
#                              tot_B["cost"] / len(page_indices),
#                              lambda x: f"${x:.4f}")

# _row("DOCX size (KB)",     out_A["size_kb"],    out_B["size_kb"])
# _row("paragraphs",         out_A["paragraphs"], out_B["paragraphs"])
# _row("tables",             out_A["tables"],     out_B["tables"])
# _row("pictures",           out_A["pictures"],   out_B["pictures"])
# _row("file",               out_A["path"] or "—", out_B["path"] or "—")

# print(f"\n{'═'*98}")
# print(" Open both DOCXs in Word for a visual spot-check — the real quality signal.")
# print(f"{'═'*98}")


## Block-aware Qwen formatter + sidecar JSON

Takes Method A's raw per-block output (`page_dicts_A`) and asks **Qwen 2.5 7B Instruct** 
to format the content **within each block**, without crossing block boundaries.

Key design: Qwen receives block-tagged input:

```
<<<BLOCK id=p1_B0 type=Title>>>
THREE YEAR B.A. DEGREE EXAMINATION
CHOICE BASED CREDIT SYSTEM
<<<END>>>

<<<BLOCK id=p1_B5 type=Text>>>
1. वाक्यं रसात्मकम् काव्यम्.
2. खंड काव्यम्.
3. स्थायी भावलु.
<<<END>>>
```

Qwen reformats CONTENT WITHIN each block (adds `#`/`##` for headings, 
`-` for options, blank lines between paragraphs, strips page furniture). 
It **cannot** merge, split, create, or delete blocks, and it **cannot** 
move content between blocks.

The output is written to a DOCX AND a sidecar JSON file:

```
{stem}__A_qwen.docx         ← readable Word document
{stem}__A_qwen.blocks.json  ← mapping block_id → {page, bbox, docx_paragraph_range}
```

The sidecar makes the review workflow possible — tap a word in the PDF → 
UI finds overlapping bbox → finds the block_id → finds which paragraph(s) 
of the DOCX contain that content.


In [ ]:
# ─── CELL 13: block-aware Qwen formatter + sidecar JSON ──────────────────
# Replaces the model-sweep cell. Qwen 2.5 7B keeps block boundaries intact.

FORMATTER_MODEL = "qwen/qwen-2.5-7b-instruct"
FMT_CHUNK_BLOCKS = 4   # how many blocks per Qwen call (keeps per-call tokens low)

# ── prompt: block-preserving reformat ───────────────────────────────────
BLOCK_FORMAT_PROMPT = """You are a document-formatting assistant. The INPUT is OCR output from a scanned document of any kind (question paper, form, table, letter, report, invoice, ...), already split into BLOCKS. Each block has a stable id and a type given by an upstream OCR engine. The type is only a hint but you have to get context from the surrounding text.

Your job: Reformat the content of all blocks for clear reading. You do NOT delete any content without explicit reasoning, example: deleting a block as its type is page number, check the information inside if its so then delete but checking is required.

INPUT FORMAT (what you will see):
Each block is delimited by `<<<BLOCK id=... type=...>>>` and `<<<END>>>` markers.

OUTPUT FORMAT (what you must emit):
Return the SAME blocks, in the SAME order, with the SAME markers (id and type preserved EXACTLY, character-for-character). Most of your task is to reformat 1% of times it would be deletion.

ABSOLUTE RULES — violations invalidate your output:
1. NEVER create a new `<<<BLOCK ...>>>` marker.
2. NEVER delete a `<<<BLOCK ...>>>` or `<<<END>>>` marker.
3. NEVER move content from one block to another block.
4. NEVER change a block's id= or type= attribute.
5. NEVER remove words, numbers, letters, question labels, or option labels from the content, EXCEPT as explicitly permitted by the REMOVAL section below.
6. Preserve every script exactly (Latin, Devanagari, Telugu, Kannada, Tamil, any indic languages). UTF-8 only.

REMOVAL — TWO LEVELS, HANDLED SEPARATELY. Default is to keep every character.

LEVEL 1 — PARTIAL REMOVAL (deleting characters, words, or lines INSIDE a block while OTHER content in the same block stays).
Allowed only when ALL of these hold:
(a) the removed text is REDUNDANT: the very same words appear again in this input, either immediately repeated (a stuttered word like "the the the", or a line printed twice in a row), or an identical stand-alone line elsewhere in this input that adds nothing new there;
(b) the copy is an echo or noise, not a separate item. Two items are NOT redundant just because they share one or two words, a common stem or ending, or the same label pattern. Options, list items, sub-questions, table cells, repeated section headings, repeated "or"-type connectors and parallel sentences are separate content even when they look alike. If ANY word, number, or label differs, they are different;
(c) deleting it loses no meaning, no item, and no structure.
Keep exactly one copy (the first) and remove only the extra copies. If unsure, keep everything.

LEVEL 2 — FULL BLOCK EMPTYING (emitting `<<<BLOCK id=... type=...>>>` immediately followed by `<<<END>>>` with NO content between them, i.e. erasing the entire body of a block).
This is MUCH STRICTER than Level 1. A full block emptying requires explicit reasoning and is allowed only when ONE of these is clearly true:
(a) the ENTIRE block, after reading its actual text, is confirmed page furniture — examples:
    - scanner watermarks: "Scanned by CamScanner", "Scanned with OKEN Scanner", similar
    - page codes: "1-2-103", "Page 5 of 10", "[1]"
    - page-turn hints: "[P.T.O.]", "Please Turn Over", "Continued on next page"
    - folio numbers in a corner
(b) OR every single word of the block is a duplicate of content kept in another block (the whole block is a stuttered duplicate of a Level 1 kind, applied to the entire block).
IMPORTANT:
- A block whose type is PageHeader / PageFooter / PageNumber / Folio is NOT automatically furniture. If its TEXT contains real content (a title, a date, a reference number, a question, a name), it is BODY CONTENT — keep it populated.
- Partial removal (Level 1) is NOT a path to full emptying. If after Level-1 removal any real content remains, keep that content in the block.
- Being unsure = KEEP THE BLOCK POPULATED. Full block emptying should be rare.

Nothing else may ever be deleted.

FORMATTING MARKERS you may add inside a block:
- `# ` prefix → big heading (for document title lines)
- `## ` prefix → medium heading (for part/section headers like "PART-A", "विभाग - ए")
- `### ` prefix → small heading
- `- ` prefix → bullet (use ONE per line for MCQ options)
- `*text*` → italic (use for marks notation like "(5 marks)" or "मार्कुलु : 5 × 5 = 25")
- Blank line between paragraphs within the same block.

TABLES — if the input block contains a table (either Bodhan tagged type=Table, or the body contains HTML tags like <table>, <tr>, <td>, <th>, or you otherwise recognise a tabular layout), emit a Markdown table in the block body. Use this exact Markdown table syntax:

| Header 1 | Header 2 | Header 3 |
| --- | --- | --- |
| cell | cell | cell |
| cell | cell | cell |

Rules for Markdown tables:
- First row = headers.
- Second row MUST be the separator row with one `---` per column, separated by `|`.
- Every data row MUST have the same number of `|`-separated cells as the header row. Pad short rows with empty cells, never merge or drop cells.
- Preserve EVERY cell's text exactly, including numbers, dashes in ranges (`2 - 3`), and punctuation. Do not reword cell content.
- If a cell in the OCR has a line break (<br/> or similar), replace it with a single space inside the Markdown cell.
- If the input is already HTML (<table>...<tr>...<td>...</td>...</tr>...</table>), convert faithfully to Markdown. Do not emit any HTML tags in your output.
- Do not surround the table with code fences.
- The table is the ONLY content of that block if the block's body was entirely a table in the input.
- If and only the text inside the block is like footer that is not required, like "scanned by "(or anything like that), you may remove it as a Level 1 redundant line. Otherwise, keep all text in the table block.

STRUCTURING GUIDELINES — apply within each block:
- If a block has multiple numbered items packed on one line ("1. ... 2. ... 3. ..."), split them into separate paragraphs.
- If a block contains an MCQ question with options (a), (b), (क), (अ): put the stem on its own paragraph, then each option on its own `- ` line.

Output the reformatted blocks and NOTHING else. No preamble like "Here is the formatted text:", no code fences, no comments. Just the blocks."""


# ── collect blocks across all pages (page_dicts_A → flat list) ──────────
def _collect_blocks_from_page_dicts(page_dicts):
    """Flatten page_dicts into a list of block dicts with stable ids."""
    out = []
    for page in page_dicts:
        for region in page["regions"]:
            # Pictures are passed through unchanged, no Qwen involvement.
            # We tag them so the DOCX builder knows how to render them.
            if region.get("type") == "picture":
                out.append({
                    "block_id":    region["block_id"],
                    "page":        region["page"],
                    "bodhan_type": region.get("bodhan_type", "Picture"),
                    "bbox_xyxy":   region.get("bbox_xyxy"),
                    "image_path":  region.get("image_path"),
                    "text":        "",
                    "is_picture":  True,
                })
                continue
            out.append({
                "block_id":    region["block_id"],
                "page":        region["page"],
                "bodhan_type": region.get("bodhan_type", "Text"),
                "bbox_xyxy":   region.get("bbox_xyxy"),
                "text":        region.get("text", "") or "",
                "is_picture":  False,
            })
    return out


# ── serialise/deserialise block-tagged text for Qwen I/O ────────────────
_BLOCK_TAG_RE = re.compile(
    r"<<<\s*BLOCK\s+id\s*=\s*(?P<id>[^\s>]+)\s+type\s*=\s*(?P<type>[^\s>]+)\s*>>>\s*\n?"
    r"(?P<body>.*?)"
    r"<<<\s*END\s*>>>",
    re.S | re.I,
)

def _serialize_blocks_for_qwen(blocks):
    """Turn a list of (text) blocks into one tagged string Qwen can consume."""
    parts = []
    for b in blocks:
        if b.get("is_picture"): continue  # pictures handled Python-side
        tid = b["block_id"]
        ttype = b["bodhan_type"]
        body = (b.get("text") or "").strip()
        parts.append(f"<<<BLOCK id={tid} type={ttype}>>>\n{body}\n<<<END>>>")
    return "\n\n".join(parts)

def _deserialize_qwen_output(text):
    """Pull block bodies back out of Qwen's output. Returns dict: id → body."""
    out = {}
    for m in _BLOCK_TAG_RE.finditer(text):
        out[m.group("id")] = m.group("body").strip()
    return out


# ── call Qwen on a chunk of blocks, with a strict integrity check ───────
def _call_qwen_on_blocks(blocks_chunk, timeout=180, _depth=0):
    """Call Qwen on a list of blocks and return {block_id: body}.
    If some blocks are missing in Qwen's response, retry those individually
    (bounded recursion). Anything still missing falls back to raw text,
    with explicit logging at every step so the retry outcome is visible.
    """
    indent = "  " * _depth
    tagged = _serialize_blocks_for_qwen(blocks_chunk)
    if not tagged.strip():
        return {}
    body = {"model": FORMATTER_MODEL, "temperature": 0,
            "messages": [{"role": "user",
                          "content": BLOCK_FORMAT_PROMPT + "\n\nINPUT:\n" + tagged}]}
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
                               "Content-Type": "application/json"},
                      json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
    raw    = data["choices"][0]["message"]["content"]
    finish = data["choices"][0].get("finish_reason", "?")
    parsed = _deserialize_qwen_output(raw)

    expected = {b["block_id"] for b in blocks_chunk if not b.get("is_picture")}
    got      = set(parsed.keys())
    missing  = expected - got
    extra    = got - expected

    if extra:
        print(f"    {indent}⚠ Qwen emitted unknown block id(s): {sorted(extra)[:5]} — ignoring")
        for e in extra: parsed.pop(e, None)

    # Retry only when chunk has >1 block AND we're not already too deep (avoid loops).
    if missing and len(blocks_chunk) > 1 and _depth < 2:
        retry_blocks = [b for b in blocks_chunk if b["block_id"] in missing and not b.get("is_picture")]
        retry_ids    = [b["block_id"] for b in retry_blocks]
        print(f"    {indent}↻ retrying {len(retry_blocks)} missing block(s): {retry_ids}  "
              f"(finish_reason={finish!r}, depth={_depth})")
        try:
            retry_parsed = _call_qwen_on_blocks(retry_blocks, _depth=_depth + 1)
            recovered    = set(retry_parsed.keys()) & set(retry_ids)
            still_miss   = set(retry_ids) - set(retry_parsed.keys())
            print(f"    {indent}↻ retry recovered {len(recovered)}/{len(retry_ids)} block(s): "
                  f"{sorted(recovered)}")
            if still_miss:
                print(f"    {indent}↻ STILL MISSING after retry: {sorted(still_miss)} "
                      f"— using RAW text for those")
            for bid, body_text in retry_parsed.items():
                parsed[bid] = body_text
            # explicit raw fallback for anything still missing after retry
            for b in blocks_chunk:
                if b["block_id"] in still_miss and not b.get("is_picture"):
                    parsed[b["block_id"]] = (b.get("text") or "").strip()
        except Exception as e:
            print(f"    {indent}↻ retry raised: {type(e).__name__}: {str(e)[:150]} "
                  f"— using RAW text for all missing")
            for b in blocks_chunk:
                if b["block_id"] in missing and not b.get("is_picture"):
                    parsed[b["block_id"]] = (b.get("text") or "").strip()

    elif missing:
        # No retry possible — single-block chunk or max depth. Raw fallback.
        print(f"    {indent}⚠ Qwen omitted {len(missing)} marker(s) and no retry possible "
              f"(len={len(blocks_chunk)}, depth={_depth}): {sorted(missing)} — using RAW text")
        for b in blocks_chunk:
            if b["block_id"] in missing and not b.get("is_picture"):
                parsed[b["block_id"]] = (b.get("text") or "").strip()

    return parsed


# ── run Qwen, chunk by chunk ────────────────────────────────────────────
blocks_A = _collect_blocks_from_page_dicts(page_dicts_A)
print(f"total blocks: {len(blocks_A)} "
      f"({sum(1 for b in blocks_A if not b['is_picture'])} text, "
      f"{sum(1 for b in blocks_A if b['is_picture'])} picture)")

text_blocks = [b for b in blocks_A if not b["is_picture"]]
chunks = [text_blocks[i:i+FMT_CHUNK_BLOCKS]
          for i in range(0, len(text_blocks), FMT_CHUNK_BLOCKS)]
print(f"qwen chunks: {len(chunks)} of up to {FMT_CHUNK_BLOCKS} blocks each")

set_stage("A_format", "qwen_blocks")
formatted_by_id = {}
qwen_failures = []
for ci, chunk in enumerate(chunks, 1):
    try:
        print(f"  chunk {ci}/{len(chunks)} ({len(chunk)} blocks) …")
        formatted_by_id.update(_call_qwen_on_blocks(chunk))
    except Exception as e:
        print(f"    HTTP/parse error: {type(e).__name__}: {str(e)[:200]}")
        print(f"    keeping RAW text for the {len(chunk)} blocks in this chunk")
        for b in chunk:
            formatted_by_id[b["block_id"]] = (b.get("text") or "").strip()
        qwen_failures.append(ci)
set_stage("-", "-")

# attach formatted_text to each block for the DOCX builder below
for b in blocks_A:
    if b.get("is_picture"): continue
    b["formatted_text"] = formatted_by_id.get(b["block_id"], b.get("text", ""))


# ── DOCX builder with per-block paragraph tracking ──────────────────────
# No type-based filtering. Every block that Qwen returned (or that fell back
# to raw text) is rendered. Only truly empty bodies produce no paragraphs.
# Markup emitted by Qwen (#/##/###/- /*italic*/Markdown tables) is turned
# into real Word styles and real Word tables here.

from docx.enum.text import WD_ALIGN_PARAGRAPH


# NOTE: build_docx() in Cell 4 already defines _add_table(doc, rows) which
# makes a real Word table styled "Light Grid Accent 1" and applies the font.
# We reuse it here for Qwen-emitted Markdown tables.

_MD_TABLE_SEP_RE = re.compile(r"^\s*\|?\s*(?::?-{3,}:?\s*\|\s*)+:?-{3,}:?\s*\|?\s*$")

def _split_md_row(line):
    """Split a Markdown table row like `| a | b | c |` into cell list ['a','b','c']."""
    s = line.strip()
    if s.startswith("|"): s = s[1:]
    if s.endswith("|"):   s = s[:-1]
    return [c.strip() for c in s.split("|")]

def _extract_md_tables(text):
    """Scan Qwen's output for Markdown tables.
    Returns a list of segments: ('text', str) or ('table', [[cell,...], ...]).
    Segments appear in document order and together reconstruct the full body.
    """
    lines = (text or "").split("\n")
    segments = []
    buf_text = []
    i = 0
    while i < len(lines):
        # Try to detect a Markdown table starting at line i: header | sep | data...
        if (i + 1 < len(lines)
                and "|" in lines[i]
                and _MD_TABLE_SEP_RE.match(lines[i + 1])):
            header = _split_md_row(lines[i])
            # data rows start at i+2 and continue while they look like table rows
            rows = [header]
            j = i + 2
            while j < len(lines) and "|" in lines[j] and lines[j].strip():
                # stop if we accidentally hit another separator (rare)
                if _MD_TABLE_SEP_RE.match(lines[j]): break
                rows.append(_split_md_row(lines[j]))
                j += 1
            # normalise column counts
            ncols = max(len(r) for r in rows)
            rows = [r + [""] * (ncols - len(r)) for r in rows]
            # flush pending text buffer first
            if buf_text:
                segments.append(("text", "\n".join(buf_text).strip("\n")))
                buf_text = []
            segments.append(("table", rows))
            i = j
            continue
        buf_text.append(lines[i])
        i += 1
    if buf_text:
        tail = "\n".join(buf_text).strip("\n")
        if tail: segments.append(("text", tail))
    return segments


def _render_text_segment(docx, text):
    """Render a non-table text segment. Converts Qwen's markdown markers:
       `# `/`## `/`### ` → Word Heading 1/2/3,
       `- ` → List Bullet,
       `*text*` → italic run,
       everything else → Normal paragraph.
       Returns the list of paragraph indices produced (in document order).
    """
    produced = []
    for raw_para in re.split(r"\n\s*\n", (text or "").strip()):
        lines = [l.rstrip() for l in raw_para.split("\n") if l.strip()]
        if not lines: continue
        for ln in lines:
            stripped = ln.lstrip()
            if stripped.startswith("### "):
                p = docx.add_heading(stripped[4:], level=3)
            elif stripped.startswith("## "):
                p = docx.add_heading(stripped[3:], level=2)
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            elif stripped.startswith("# "):
                p = docx.add_heading(stripped[2:], level=1)
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            elif stripped.startswith("- "):
                p = docx.add_paragraph(stripped[2:], style="List Bullet")
            elif (stripped.startswith("*") and stripped.endswith("*")
                  and len(stripped) >= 3 and not stripped.startswith("**")):
                p = docx.add_paragraph()
                r = p.add_run(stripped.strip("*")); r.italic = True
            else:
                p = docx.add_paragraph(ln)
            for run in p.runs: _apply_font(run)
            produced.append(len(docx.paragraphs) - 1)
    return produced


def _render_formatted_text(docx, text):
    """Parse Qwen's marker-formatted text and emit DOCX paragraphs/tables.
    Returns the list of paragraph indices this call produced (in body order).
    Note: when a Markdown table is rendered, we record the paragraph index of
    the row immediately AFTER the table so the sidecar's paragraph_range still
    maps to something addressable. The table itself lives between paragraphs.
    """
    produced = []
    segments = _extract_md_tables(text)
    for kind, payload in segments:
        if kind == "text":
            produced.extend(_render_text_segment(docx, payload))
        elif kind == "table":
            # Reuse the Cell 4 helper for proper Word tables with styling.
            try:
                _add_table(docx, payload)
            except Exception as e:
                # Fall back to rendering the raw cells as text lines so
                # we never silently lose content.
                print(f"    ⚠ table render failed, falling back to text: "
                      f"{type(e).__name__}: {str(e)[:120]}")
                flat = "\n".join(" | ".join(r) for r in payload)
                produced.extend(_render_text_segment(docx, flat))
                continue
            # record a placeholder paragraph after the table so the sidecar
            # has a valid index to point at
            p = docx.add_paragraph("")
            for run in p.runs: _apply_font(run)
            produced.append(len(docx.paragraphs) - 1)
    return produced


def build_docx_with_blocks(blocks, out_path):
    """Write blocks to DOCX and return mapping block_id -> {page, bbox, para_range}.
    No type-based filtering — every block Qwen returned (or that fell back to raw)
    is rendered. Only truly empty text produces no paragraphs.
    """
    docx = Document()
    docx.styles["Normal"].font.name = _FONT_NAME
    docx.styles["Normal"].font.size = Pt(11)

    block_meta = {}
    for b in blocks:
        bid = b["block_id"]

        # pictures: render the cropped image
        if b.get("is_picture") and b.get("image_path"):
            try:
                docx.add_picture(b["image_path"], width=Inches(5.5))
                idx = len(docx.paragraphs) - 1
                block_meta[bid] = {
                    "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
                    "bodhan_type": b["bodhan_type"], "rendered": True,
                    "docx_paragraph_range": [idx, idx],
                }
            except Exception as e:
                docx.add_paragraph(f"[picture failed: {e}]")
                block_meta[bid] = {
                    "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
                    "bodhan_type": b["bodhan_type"], "rendered": False,
                    "reason": f"picture_error: {e}",
                    "docx_paragraph_range": None,
                }
            continue

        text = sanitize(b.get("formatted_text") or "")
        if not text:
            block_meta[bid] = {
                "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
                "bodhan_type": b["bodhan_type"], "rendered": False,
                "reason": "empty_text",
                "docx_paragraph_range": None,
            }
            continue

        produced = _render_formatted_text(docx, text)
        if produced:
            block_meta[bid] = {
                "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
                "bodhan_type": b["bodhan_type"], "rendered": True,
                "docx_paragraph_range": [produced[0], produced[-1]],
            }
        else:
            block_meta[bid] = {
                "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
                "bodhan_type": b["bodhan_type"], "rendered": False,
                "reason": "no_paragraphs_emitted",
                "docx_paragraph_range": None,
            }

    docx.save(str(out_path))
    return str(out_path), block_meta


# ── run the builder ────────────────────────────────────────────────────
set_stage("A_format", "build_docx")
docx_qwen_path = METHOD_A_DIR / f"{stem}__A_qwen.docx"
try:
    final_qwen, block_meta = build_docx_with_blocks(blocks_A, docx_qwen_path)
except PermissionError:
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    docx_qwen_path = docx_qwen_path.with_name(f"{docx_qwen_path.stem}_{ts}.docx")
    final_qwen, block_meta = build_docx_with_blocks(blocks_A, docx_qwen_path)
    print(f"  (original was locked, saved as {docx_qwen_path.name})")
print(f"\nsaved {final_qwen}")

# ── sidecar JSON ───────────────────────────────────────────────────────
sidecar = {
    "pdf_path":        str(PDF_PATH),
    "docx_path":       str(final_qwen),
    "page_count":      total,
    "pages_processed": [p + 1 for p in page_indices],
    "formatter_model": FORMATTER_MODEL,
    "qwen_failures":   qwen_failures,  # chunk indices (1-based) that fell back to raw text
    "blocks":          block_meta,
}
sidecar_path = Path(final_qwen).with_suffix("").with_suffix(".blocks.json")
sidecar_path.write_text(json.dumps(sidecar, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved {sidecar_path}")

# ── compact stats ──────────────────────────────────────────────────────
rendered = sum(1 for m in block_meta.values() if m["rendered"])
empty    = sum(1 for m in block_meta.values() if m.get("reason") == "empty_text")

qwen_calls = [c for c in CALLS
              if c["method"] == "A_format" and c["engine"] != "_wallclock_"]
cost = sum((c["cost"] or 0) for c in qwen_calls)
tin  = sum(c["tok_in"]  for c in qwen_calls)
tout = sum(c["tok_out"] for c in qwen_calls)
sec  = sum(c["seconds"] for c in qwen_calls)

print(f"\nblocks:     {len(block_meta)} total")
print(f"  rendered: {rendered} \u2192 DOCX paragraphs")
print(f"  empty:    {empty} (truly empty text, nothing to render)")
print(f"qwen:       {len(qwen_calls)} call(s), {tin:,}/{tout:,} tok, {sec:.1f}s, ${cost:.5f}")
print(f"            {len(qwen_failures)} chunk(s) fell back to raw text: {qwen_failures}")
set_stage("-", "-")

total blocks: 200 (192 text, 8 picture)
qwen chunks: 48 of up to 4 blocks each
  chunk 1/48 (4 blocks) …
  chunk 2/48 (4 blocks) …
  chunk 3/48 (4 blocks) …
  chunk 4/48 (4 blocks) …
  chunk 5/48 (4 blocks) …
  chunk 6/48 (4 blocks) …
  chunk 7/48 (4 blocks) …
  chunk 8/48 (4 blocks) …
    ↻ retrying 1 missing block(s): ['p4_B1']  (finish_reason='stop', depth=0)
    ↻ retry recovered 1/1 block(s): ['p4_B1']
  chunk 9/48 (4 blocks) …
  chunk 10/48 (4 blocks) …
  chunk 11/48 (4 blocks) …
  chunk 12/48 (4 blocks) …
  chunk 13/48 (4 blocks) …
  chunk 14/48 (4 blocks) …
    ↻ retrying 3 missing block(s): ['p6_B6', 'p6_B7', 'p6_B8']  (finish_reason='stop', depth=0)
    ↻ retry recovered 3/3 block(s): ['p6_B6', 'p6_B7', 'p6_B8']
  chunk 15/48 (4 blocks) …
  chunk 16/48 (4 blocks) …
  chunk 17/48 (4 blocks) …
  chunk 18/48 (4 blocks) …
  chunk 19/48 (4 blocks) …
  chunk 20/48 (4 blocks) …
  chunk 21/48 (4 blocks) …
  chunk 22/48 (4 blocks) …
  chunk 23/48 (4 blocks) …
  chunk 24/48 (4 blocks) 

In [71]:
# # ─── CELL 13: block-aware Qwen formatter + sidecar JSON ──────────────────
# # Replaces the model-sweep cell. Qwen 2.5 7B keeps block boundaries intact.

# FORMATTER_MODEL = "qwen/qwen-2.5-7b-instruct"
# FMT_CHUNK_BLOCKS = 4   # how many blocks per Qwen call (keeps per-call tokens low)

# # ── prompt: block-preserving reformat ───────────────────────────────────
# # BLOCK_FORMAT_PROMPT = """You are a document-formatting assistant. The INPUT is OCR output from a scanned document of any kind (question paper, form, table, letter, report, invoice, ...), already split into BLOCKS. Each block has a stable id and a type given by an upstream OCR engine. The type is only a hint but you have to get context from the surrounding text.

# # Your job: Reformat the content of all blocks for clear reading. You do NOT delete any content without explicit reasoning, example: deleting a block as its type is page number, check the information inside if its so then delete but checking is required.

# # INPUT FORMAT (what you will see):
# # Each block is delimited by `<<<BLOCK id=... type=...>>>` and `<<<END>>>` markers.

# # OUTPUT FORMAT (what you must emit):
# # Return the SAME blocks, in the SAME order, with the SAME markers (id and type preserved EXACTLY, character-for-character). Most of your task is to reformat 1% of times it would be deletion.

# # ABSOLUTE RULES — violations invalidate your output:
# # 1. NEVER create a new `<<<BLOCK ...>>>` marker.
# # 2. NEVER delete a `<<<BLOCK ...>>>` or `<<<END>>>` marker.
# # 3. NEVER move content from one block to another block.
# # 4. NEVER change a block's id= or type= attribute.
# # 5. NEVER add new words, numbers, letters, or option labels.
# # 6. NEVER translate, paraphrase, or correct spelling.
# # 7. Preserve every script exactly (Latin, Devanagari, Telugu, Kannada, Tamil, any indic languages). UTF-8 only.

# # REMOVAL — TWO LEVELS, HANDLED SEPARATELY. Default is to keep every character.

# # LEVEL 1 — PARTIAL REMOVAL (deleting characters, words, or lines INSIDE a block while OTHER content in the same block stays).
# # Allowed only when ALL of these hold:
# # (a) the removed text is REDUNDANT: the very same words appear again in this input, either immediately repeated (a stuttered word like "the the the", or a line printed twice in a row), or an identical stand-alone line elsewhere in this input that adds nothing new there;
# # (b) the copy is an echo or noise, not a separate item. Two items are NOT redundant just because they share one or two words, a common stem or ending, or the same label pattern. Options, list items, sub-questions, table cells, repeated section headings, repeated "or"-type connectors and parallel sentences are separate content even when they look alike. If ANY word, number, or label differs, they are different;
# # (c) deleting it loses no meaning, no item, and no structure.
# # Keep exactly one copy (the first) and remove only the extra copies. If unsure, keep everything.

# # LEVEL 2 — FULL BLOCK EMPTYING (emitting `<<<BLOCK id=... type=...>>>` immediately followed by `<<<END>>>` with NO content between them, i.e. erasing the entire body of a block).
# # This is MUCH STRICTER than Level 1. A full block emptying requires explicit reasoning and is allowed only when ONE of these is clearly true:
# # (a) the ENTIRE block, after reading its actual text, is confirmed page furniture — examples:
# #     - scanner watermarks: "Scanned by CamScanner", "Scanned with OKEN Scanner", similar
# #     - page codes: "1-2-103", "Page 5 of 10", "[1]"
# #     - page-turn hints: "[P.T.O.]", "Please Turn Over", "Continued on next page"
# #     - folio numbers in a corner
# # (b) OR every single word of the block is a duplicate of content kept in another block (the whole block is a stuttered duplicate of a Level 1 kind, applied to the entire block).
# # IMPORTANT:
# # - A block whose type is PageHeader / PageFooter / PageNumber / Folio is NOT automatically furniture. If its TEXT contains real content (a title, a date, a reference number, a question, a name), it is BODY CONTENT — keep it populated.
# # - Partial removal (Level 1) is NOT a path to full emptying. If after Level-1 removal any real content remains, keep that content in the block.
# # - Being unsure = KEEP THE BLOCK POPULATED. Full block emptying should be rare.

# # Nothing else may ever be deleted.

# # FORMATTING MARKERS you may add inside a block:
# # - `# ` prefix → big heading (for document title lines)
# # - `## ` prefix → medium heading (for part/section headers like "PART-A", "विभाग - ए")
# # - `### ` prefix → small heading
# # - `- ` prefix → bullet (use ONE per line for MCQ options)
# # - `---` on its own line → centered separator (use for "OR", "లేదా", "अथवा")
# # - `*text*` → italic (use for marks notation like "(5 marks)" or "मार्कुलु : 5 × 5 = 25")
# # - Blank line between paragraphs within the same block.

# # STRUCTURING GUIDELINES — apply within each block:
# # - If a block has multiple numbered items packed on one line ("1. ... 2. ... 3. ..."), split them into separate paragraphs.
# # - If a block contains an MCQ question with options (a), (b), (क), (अ): put the stem on its own paragraph, then each option on its own `- ` line.

# # Output the reformatted blocks and NOTHING else. No preamble like "Here is the formatted text:", no code fences, no comments. Just the blocks."""

# BLOCK_FORMAT_PROMPT = """You are a text-reflow assistant. The INPUT is OCR text from a scanned document, split into blocks. Each block is delimited by `<<<BLOCK id=... type=...>>>` and `<<<END>>>` markers.

# Your SINGLE job: inside each block, insert blank lines between logical paragraphs to make the text easier to read. Do NOTHING else.

# OUTPUT: Return the SAME blocks, in the SAME order, with the SAME `<<<BLOCK id=... type=...>>>` and `<<<END>>>` markers (id and type preserved character-for-character). Inside each block, you may insert blank lines. That is the only change you may make.

# HARD RULES — any violation invalidates your output:

# 1. BLOCK MARKERS: never create, delete, modify, re-order, or merge `<<<BLOCK>>>` or `<<<END>>>` markers. Every input block must have exactly one corresponding output block with the same id and type.

# 2. CONTENT PRESERVATION — every character of input content must appear in the output:
#    - every word, number, letter, label, punctuation mark, bracket, dot, dash
#    - scanner watermarks, page codes, page-turn hints, running headers/footers
#    - separator words in any language and any script
#    - repeated, duplicate, or similar-looking text
#    - anything that looks like noise, OCR error, or furniture
#    KEEP ALL OF IT. Not one character may be removed, from any block, for any reason, under any circumstances.

# 3. NO SUBSTITUTIONS: do not replace any word, phrase, or character with a symbol, marker, dash, underscore, or any other character. Whatever was in the input must be in the output, byte for byte.

# 4. NO MARKUP: do not add any Markdown or formatting marker. No `#`, no `##`, no `###`, no `-`, no `*`, no `---`, no `___`, no HTML, no XML, no backticks. Output is plain text that differs from input ONLY in added blank lines.

# 5. NO TRANSLATION, NO PARAPHRASING, NO SPELLING FIXES: preserve every script exactly (Latin, Devanagari, Telugu, Kannada, Tamil, any other). UTF-8 only.

# 6. NO MOVING CONTENT: text in block X must stay in block X. Never move text between blocks.

# 7. NO ADDITIONS: do not add any word, label, heading, title, explanation, preamble, comment, or code fence.

# WHAT YOU MAY DO (only this):
# - Insert blank lines between logical paragraphs within a block.
# - Where multiple logical items were run together on one line, put each on its own line while preserving every character including original numbering, labels, and punctuation.

# If you are ever unsure whether something counts as content or noise: KEEP IT.
# NO MAJOR DELTIONS ARE EVER ALLOWED. YOU ARE ONLY A TEXT FORMATER. DO NOT DELETE ANYTHING.
# Output the reformatted blocks and nothing else. No preamble, no code fences, no explanations."""


# # ── collect blocks across all pages (page_dicts_A → flat list) ──────────
# def _collect_blocks_from_page_dicts(page_dicts):
#     """Flatten page_dicts into a list of block dicts with stable ids."""
#     out = []
#     for page in page_dicts:
#         for region in page["regions"]:
#             # Pictures are passed through unchanged, no Qwen involvement.
#             # We tag them so the DOCX builder knows how to render them.
#             if region.get("type") == "picture":
#                 out.append({
#                     "block_id":    region["block_id"],
#                     "page":        region["page"],
#                     "bodhan_type": region.get("bodhan_type", "Picture"),
#                     "bbox_xyxy":   region.get("bbox_xyxy"),
#                     "image_path":  region.get("image_path"),
#                     "text":        "",
#                     "is_picture":  True,
#                 })
#                 continue
#             out.append({
#                 "block_id":    region["block_id"],
#                 "page":        region["page"],
#                 "bodhan_type": region.get("bodhan_type", "Text"),
#                 "bbox_xyxy":   region.get("bbox_xyxy"),
#                 "text":        region.get("text", "") or "",
#                 "is_picture":  False,
#             })
#     return out


# # ── serialise/deserialise block-tagged text for Qwen I/O ────────────────
# _BLOCK_TAG_RE = re.compile(
#     r"<<<\s*BLOCK\s+id\s*=\s*(?P<id>[^\s>]+)\s+type\s*=\s*(?P<type>[^\s>]+)\s*>>>\s*\n?"
#     r"(?P<body>.*?)"
#     r"<<<\s*END\s*>>>",
#     re.S | re.I,
# )

# def _serialize_blocks_for_qwen(blocks):
#     """Turn a list of (text) blocks into one tagged string Qwen can consume."""
#     parts = []
#     for b in blocks:
#         if b.get("is_picture"): continue  # pictures handled Python-side
#         tid = b["block_id"]
#         ttype = b["bodhan_type"]
#         body = (b.get("text") or "").strip()
#         parts.append(f"<<<BLOCK id={tid} type={ttype}>>>\n{body}\n<<<END>>>")
#     return "\n\n".join(parts)

# def _deserialize_qwen_output(text):
#     """Pull block bodies back out of Qwen's output. Returns dict: id → body."""
#     out = {}
#     for m in _BLOCK_TAG_RE.finditer(text):
#         out[m.group("id")] = m.group("body").strip()
#     return out


# # ── call Qwen on a chunk of blocks, with a strict integrity check ───────
# def _call_qwen_on_blocks(blocks_chunk, timeout=180, _depth=0):
#     """Call Qwen on a list of blocks and return {block_id: body}.
#     If some blocks are missing in Qwen's response, retry those individually
#     (bounded recursion). Anything still missing falls back to raw text,
#     with explicit logging at every step so the retry outcome is visible.
#     """
#     indent = "  " * _depth
#     tagged = _serialize_blocks_for_qwen(blocks_chunk)
#     if not tagged.strip():
#         return {}
#     body = {"model": FORMATTER_MODEL, "temperature": 0,
#             "messages": [{"role": "user",
#                           "content": BLOCK_FORMAT_PROMPT + "\n\nINPUT:\n" + tagged}]}
#     r = requests.post(OPENROUTER_URL,
#                       headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
#                                "Content-Type": "application/json"},
#                       json=body, timeout=timeout)
#     r.raise_for_status()
#     data = r.json()
#     if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
#     raw    = data["choices"][0]["message"]["content"]
#     finish = data["choices"][0].get("finish_reason", "?")
#     parsed = _deserialize_qwen_output(raw)

#     expected = {b["block_id"] for b in blocks_chunk if not b.get("is_picture")}
#     got      = set(parsed.keys())
#     missing  = expected - got
#     extra    = got - expected

#     if extra:
#         print(f"    {indent}⚠ Qwen emitted unknown block id(s): {sorted(extra)[:5]} — ignoring")
#         for e in extra: parsed.pop(e, None)

#     # Retry only when chunk has >1 block AND we're not already too deep (avoid loops).
#     if missing and len(blocks_chunk) > 1 and _depth < 2:
#         retry_blocks = [b for b in blocks_chunk if b["block_id"] in missing and not b.get("is_picture")]
#         retry_ids    = [b["block_id"] for b in retry_blocks]
#         print(f"    {indent}↻ retrying {len(retry_blocks)} missing block(s): {retry_ids}  "
#               f"(finish_reason={finish!r}, depth={_depth})")
#         try:
#             retry_parsed = _call_qwen_on_blocks(retry_blocks, _depth=_depth + 1)
#             recovered    = set(retry_parsed.keys()) & set(retry_ids)
#             still_miss   = set(retry_ids) - set(retry_parsed.keys())
#             print(f"    {indent}↻ retry recovered {len(recovered)}/{len(retry_ids)} block(s): "
#                   f"{sorted(recovered)}")
#             if still_miss:
#                 print(f"    {indent}↻ STILL MISSING after retry: {sorted(still_miss)} "
#                       f"— using RAW text for those")
#             for bid, body_text in retry_parsed.items():
#                 parsed[bid] = body_text
#             # explicit raw fallback for anything still missing after retry
#             for b in blocks_chunk:
#                 if b["block_id"] in still_miss and not b.get("is_picture"):
#                     parsed[b["block_id"]] = (b.get("text") or "").strip()
#         except Exception as e:
#             print(f"    {indent}↻ retry raised: {type(e).__name__}: {str(e)[:150]} "
#                   f"— using RAW text for all missing")
#             for b in blocks_chunk:
#                 if b["block_id"] in missing and not b.get("is_picture"):
#                     parsed[b["block_id"]] = (b.get("text") or "").strip()

#     elif missing:
#         # No retry possible — single-block chunk or max depth. Raw fallback.
#         print(f"    {indent}⚠ Qwen omitted {len(missing)} marker(s) and no retry possible "
#               f"(len={len(blocks_chunk)}, depth={_depth}): {sorted(missing)} — using RAW text")
#         for b in blocks_chunk:
#             if b["block_id"] in missing and not b.get("is_picture"):
#                 parsed[b["block_id"]] = (b.get("text") or "").strip()

#     return parsed


# # ── run Qwen, chunk by chunk ────────────────────────────────────────────
# blocks_A = _collect_blocks_from_page_dicts(page_dicts_A)
# print(f"total blocks: {len(blocks_A)} "
#       f"({sum(1 for b in blocks_A if not b['is_picture'])} text, "
#       f"{sum(1 for b in blocks_A if b['is_picture'])} picture)")

# text_blocks = [b for b in blocks_A if not b["is_picture"]]
# chunks = [text_blocks[i:i+FMT_CHUNK_BLOCKS]
#           for i in range(0, len(text_blocks), FMT_CHUNK_BLOCKS)]
# print(f"qwen chunks: {len(chunks)} of up to {FMT_CHUNK_BLOCKS} blocks each")

# set_stage("A_format", "qwen_blocks")
# formatted_by_id = {}
# qwen_failures = []
# for ci, chunk in enumerate(chunks, 1):
#     try:
#         print(f"  chunk {ci}/{len(chunks)} ({len(chunk)} blocks) …")
#         formatted_by_id.update(_call_qwen_on_blocks(chunk))
#     except Exception as e:
#         print(f"    HTTP/parse error: {type(e).__name__}: {str(e)[:200]}")
#         print(f"    keeping RAW text for the {len(chunk)} blocks in this chunk")
#         for b in chunk:
#             formatted_by_id[b["block_id"]] = (b.get("text") or "").strip()
#         qwen_failures.append(ci)
# set_stage("-", "-")

# # attach formatted_text to each block for the DOCX builder below
# for b in blocks_A:
#     if b.get("is_picture"): continue
#     b["formatted_text"] = formatted_by_id.get(b["block_id"], b.get("text", ""))


# # ── DOCX builder with per-block paragraph tracking ──────────────────────
# from docx.enum.text import WD_ALIGN_PARAGRAPH

# _FURNITURE_BODHAN = {"PageHeader", "PageFooter", "PageNumber", "Folio", "Header", "Footer"}

# def _render_formatted_text(docx, text):
#     """Parse Qwen's marker-formatted text and emit DOCX paragraphs.
#     Returns the list of paragraph indices this call produced (in body order)."""
#     produced = []
#     for raw_para in re.split(r"\n\s*\n", (text or "").strip()):
#         lines = [l.rstrip() for l in raw_para.split("\n") if l.strip()]
#         if not lines: continue

#         # separator
#         if len(lines) == 1 and lines[0].strip() in ("---", "***", "___"):
#             p = docx.add_paragraph("\u2500" * 20)
#             p.alignment = WD_ALIGN_PARAGRAPH.CENTER
#             for run in p.runs: _apply_font(run)
#             produced.append(len(docx.paragraphs) - 1)
#             continue

#         # multi-line bullet block
#         if all(l.lstrip().startswith("- ") for l in lines):
#             for l in lines:
#                 p = docx.add_paragraph(l.lstrip()[2:], style="List Bullet")
#                 for run in p.runs: _apply_font(run)
#                 produced.append(len(docx.paragraphs) - 1)
#             continue

#         # per-line (single-line paragraphs or mixed)
#         for ln in lines:
#             stripped = ln.lstrip()
#             if stripped.startswith("### "):
#                 p = docx.add_heading(stripped[4:], level=3)
#             elif stripped.startswith("## "):
#                 p = docx.add_heading(stripped[3:], level=2)
#                 p.alignment = WD_ALIGN_PARAGRAPH.CENTER
#             elif stripped.startswith("# "):
#                 p = docx.add_heading(stripped[2:], level=1)
#                 p.alignment = WD_ALIGN_PARAGRAPH.CENTER
#             elif stripped.startswith("- "):
#                 p = docx.add_paragraph(stripped[2:], style="List Bullet")
#             elif stripped.startswith("*") and stripped.endswith("*") and len(stripped) > 2:
#                 p = docx.add_paragraph()
#                 r = p.add_run(stripped.strip("*")); r.italic = True; r.font.size = Pt(10)
#             else:
#                 p = docx.add_paragraph(ln)
#             for run in p.runs: _apply_font(run)
#             produced.append(len(docx.paragraphs) - 1)
#     return produced

# def build_docx_with_blocks(blocks, out_path):
#     """Write blocks to DOCX and return mapping block_id -> {page, bbox, para_range}."""
#     docx = Document()
#     docx.styles["Normal"].font.name = _FONT_NAME
#     docx.styles["Normal"].font.size = Pt(11)

#     block_meta = {}
#     for b in blocks:
#         bid = b["block_id"]
#         # skip page furniture entirely
#         if b.get("bodhan_type") in _FURNITURE_BODHAN:
#             block_meta[bid] = {
#                 "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                 "bodhan_type": b["bodhan_type"], "rendered": False,
#                 "reason": "page_furniture",
#                 "docx_paragraph_range": None,
#             }
#             continue

#         if b.get("is_picture") and b.get("image_path"):
#             try:
#                 docx.add_picture(b["image_path"], width=Inches(5.5))
#                 idx = len(docx.paragraphs) - 1
#                 block_meta[bid] = {
#                     "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                     "bodhan_type": b["bodhan_type"], "rendered": True,
#                     "docx_paragraph_range": [idx, idx],
#                 }
#             except Exception as e:
#                 docx.add_paragraph(f"[picture failed: {e}]")
#                 block_meta[bid] = {
#                     "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                     "bodhan_type": b["bodhan_type"], "rendered": False,
#                     "reason": f"picture_error: {e}",
#                     "docx_paragraph_range": None,
#                 }
#             continue

#         text = sanitize(b.get("formatted_text") or "")
#         if not text:
#             block_meta[bid] = {
#                 "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                 "bodhan_type": b["bodhan_type"], "rendered": False,
#                 "reason": "empty_text",
#                 "docx_paragraph_range": None,
#             }
#             continue

#         produced = _render_formatted_text(docx, text)
#         if produced:
#             block_meta[bid] = {
#                 "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                 "bodhan_type": b["bodhan_type"], "rendered": True,
#                 "docx_paragraph_range": [produced[0], produced[-1]],
#             }
#         else:
#             block_meta[bid] = {
#                 "page": b["page"], "bbox_xyxy": b.get("bbox_xyxy"),
#                 "bodhan_type": b["bodhan_type"], "rendered": False,
#                 "reason": "no_paragraphs_emitted",
#                 "docx_paragraph_range": None,
#             }

#     docx.save(str(out_path))
#     return str(out_path), block_meta


# # ── run the builder ────────────────────────────────────────────────────
# set_stage("A_format", "build_docx")
# docx_qwen_path = METHOD_A_DIR / f"{stem}__A_qwen.docx"
# try:
#     final_qwen, block_meta = build_docx_with_blocks(blocks_A, docx_qwen_path)
# except PermissionError:
#     ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
#     docx_qwen_path = docx_qwen_path.with_name(f"{docx_qwen_path.stem}_{ts}.docx")
#     final_qwen, block_meta = build_docx_with_blocks(blocks_A, docx_qwen_path)
#     print(f"  (original was locked, saved as {docx_qwen_path.name})")
# print(f"\nsaved {final_qwen}")

# # ── sidecar JSON ───────────────────────────────────────────────────────
# sidecar = {
#     "pdf_path":        str(PDF_PATH),
#     "docx_path":       str(final_qwen),
#     "page_count":      total,
#     "pages_processed": [p + 1 for p in page_indices],
#     "formatter_model": FORMATTER_MODEL,
#     "qwen_failures":   qwen_failures,  # chunk indices (1-based) that fell back to raw text
#     "blocks":          block_meta,
# }
# sidecar_path = Path(final_qwen).with_suffix("").with_suffix(".blocks.json")
# sidecar_path.write_text(json.dumps(sidecar, ensure_ascii=False, indent=2), encoding="utf-8")
# print(f"saved {sidecar_path}")

# # ── compact stats ──────────────────────────────────────────────────────
# rendered = sum(1 for m in block_meta.values() if m["rendered"])
# furniture = sum(1 for m in block_meta.values() if m.get("reason") == "page_furniture")
# empty    = sum(1 for m in block_meta.values() if m.get("reason") == "empty_text")

# qwen_calls = [c for c in CALLS
#               if c["method"] == "A_format" and c["engine"] != "_wallclock_"]
# cost = sum((c["cost"] or 0) for c in qwen_calls)
# tin  = sum(c["tok_in"]  for c in qwen_calls)
# tout = sum(c["tok_out"] for c in qwen_calls)
# sec  = sum(c["seconds"] for c in qwen_calls)

# print(f"\nblocks:     {len(block_meta)} total")
# print(f"  rendered: {rendered} \u2192 DOCX paragraphs")
# print(f"  furniture:{furniture} (page codes / scanner lines, correctly omitted)")
# print(f"  empty:    {empty}")
# print(f"qwen:       {len(qwen_calls)} call(s), {tin:,}/{tout:,} tok, {sec:.1f}s, ${cost:.5f}")
# print(f"            {len(qwen_failures)} chunk(s) fell back to raw text: {qwen_failures}")
# set_stage("-", "-")

## Sidecar JSON preview

A quick look at what the review UI will consume. For each block_id we have 
the PDF page, bbox in pixel coords, Bodhan's original type, whether it 
ended up in the DOCX, and the paragraph index range.


In [72]:
# ─── CELL 15: sidecar preview ────────────────────────────────────────────
side = json.loads(Path(sidecar_path).read_text(encoding="utf-8"))
print(f"sidecar: {sidecar_path}")
print(f"  pdf:  {side['pdf_path']}")
print(f"  docx: {side['docx_path']}")
print(f"  model: {side['formatter_model']}")
print(f"  blocks: {len(side['blocks'])}")
print()

# show first few blocks of each kind
rendered  = [(bid, m) for bid, m in side["blocks"].items() if m.get("rendered")]
furniture = [(bid, m) for bid, m in side["blocks"].items() if m.get("reason") == "page_furniture"]

print("── first 5 rendered blocks ──")
for bid, m in rendered[:5]:
    rng = m.get("docx_paragraph_range")
    print(f"  {bid:<16} page {m['page']}  type={m['bodhan_type']:<14}  "
          f"paras={rng[0]}\u2013{rng[1]}  bbox={m.get('bbox_xyxy')}")

print("\n── first 3 page furniture blocks (omitted from DOCX) ──")
for bid, m in furniture[:3]:
    print(f"  {bid:<16} page {m['page']}  type={m['bodhan_type']:<14}  bbox={m.get('bbox_xyxy')}")

# simulate a review-tap lookup
print("\n── demo review lookup ──")
print("suppose the user tapped a word at (page 3, pixel 420, 650).")
tap_page, tap_x, tap_y = 3, 420, 650
hits = []
for bid, m in side["blocks"].items():
    if m["page"] != tap_page or not m.get("bbox_xyxy"): continue
    x0, y0, x1, y1 = m["bbox_xyxy"]
    if x0 <= tap_x <= x1 and y0 <= tap_y <= y1:
        hits.append((bid, m))
if hits:
    print(f"  {len(hits)} block(s) contain (page {tap_page}, {tap_x}, {tap_y}):")
    for bid, m in hits:
        print(f"    {bid}  type={m['bodhan_type']}  paras={m.get('docx_paragraph_range')}")
else:
    print(f"  no block contains (page {tap_page}, {tap_x}, {tap_y})")
    print("  (expected if page 3 wasn't processed or the tap is in blank space)")


sidecar: compare_out\A_bodhan_nemotron\eng_qp_scanned__A_qwen.blocks.json
  pdf:  pdf\eng_qp_scanned.pdf
  docx: compare_out\A_bodhan_nemotron\eng_qp_scanned__A_qwen.docx
  model: qwen/qwen-2.5-7b-instruct
  blocks: 200

── first 5 rendered blocks ──
  p1_B0            page 1  type=SectionHeader   paras=0–1  bbox=[351.6, 212.8, 1282.9, 370.7]
  p1_B1            page 1  type=SectionHeader   paras=2–3  bbox=[449.8, 430.1, 1187.0, 602.5]
  p1_B2            page 1  type=Text            paras=4–4  bbox=[340.7, 685.1, 1357.1, 779.6]
  p1_B3            page 1  type=Table           paras=5–5  bbox=[335.6, 879.9, 1368.0, 1564.1]
  p1_B4            page 1  type=Text            paras=6–6  bbox=[1186.5, 2259.5, 1632.0, 2309.1]

── first 3 page furniture blocks (omitted from DOCX) ──

── demo review lookup ──
suppose the user tapped a word at (page 3, pixel 420, 650).
  1 block(s) contain (page 3, 420, 650):
    p3_B7  type=Text  paras=[54, 63]
